# Data Cleaning and Preprocessing

## Section 1: Kaggle Environment Setup

This section checks:

- Python version
- Kaggle environment
- GPU availability
- Memory and storage
- Random seed

A GPU is optional for data cleaning but may help with OCR.

In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import psutil

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# Kaggle environment
IS_KAGGLE = Path("/kaggle").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path("input")
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python version : {sys.version.split()[0]}")
print(f"Kaggle detected: {IS_KAGGLE}")
print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Random seed    : {RANDOM_SEED}")

Python version : 3.12.13
Kaggle detected: True
Input folder   : /kaggle/input
Output folder  : /kaggle/working
Random seed    : 42


In [2]:
# GPU check
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total",
         "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"GPU             : {result.stdout.strip()}")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU             : No NVIDIA GPU detected")

GPU             : Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# Memory and storage check
GB = 1024 ** 3

memory = psutil.virtual_memory()
disk = shutil.disk_usage(OUTPUT_DIR)

print(f"Total memory    : {memory.total / GB:.2f} GB")
print(f"Available memory: {memory.available / GB:.2f} GB")
print(f"Free storage    : {disk.free / GB:.2f} GB")

Total memory    : 31.35 GB
Available memory: 30.01 GB
Free storage    : 19.49 GB


## Section 2: Install and Import Required Libraries

This section prepares the libraries needed for PDF extraction, OCR, text cleaning, duplicate detection, and dataset export.

In [4]:
%pip install -q pypdf pymupdf pdfplumber pytesseract rapidfuzz beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


In [5]:
import re
import json
import hashlib
import shutil
from pathlib import Path

import fitz
import pandas as pd
import pdfplumber
import pytesseract

from bs4 import BeautifulSoup
from PIL import Image
from pypdf import PdfReader
from rapidfuzz import fuzz
from tqdm.auto import tqdm

print("All required Python libraries imported successfully.")

All required Python libraries imported successfully.


## Section 3: Configure Dataset Paths

This section locates the source documents and creates folders for processed outputs.

In [6]:
# Display attached Kaggle datasets
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

dataset_folders = [path for path in INPUT_DIR.iterdir() if path.is_dir()]

print("Attached Kaggle datasets:")
for folder in dataset_folders:
    print(f"- {folder.name}")

Attached Kaggle datasets:
- datasets


In [7]:
# Create output folders
CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORT_DIR = OUTPUT_DIR / "exports"

for folder in [OUTPUT_DIR, CLEANED_DIR, REPORTS_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Cleaned folder : {CLEANED_DIR}")
print(f"Reports folder : {REPORTS_DIR}")
print(f"Export folder  : {EXPORT_DIR}")

Input folder   : /kaggle/input
Output folder  : /kaggle/working/tax_llm
Cleaned folder : /kaggle/working/tax_llm/cleaned_documents
Reports folder : /kaggle/working/tax_llm/reports
Export folder  : /kaggle/working/tax_llm/exports


In [8]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = [
    path
    for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
]

print(f"Supported source files found: {len(source_files)}")

for path in source_files[:20]:
    print(path)

Supported source files found: 58
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data

## Section 3: Configure Dataset Paths

This section locates the raw tax documents and creates folders for processed outputs.

In [9]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

dataset_folders = sorted(
    path for path in INPUT_DIR.iterdir() if path.is_dir()
)

print(f"Attached datasets: {len(dataset_folders)}")

for index, folder in enumerate(dataset_folders):
    print(f"{index}: {folder}")

Attached datasets: 1
0: /kaggle/input/datasets


In [10]:
if len(dataset_folders) != 1:
    raise ValueError(
        "Expected one attached dataset. Select the correct folder manually."
    )

RAW_DATA_DIR = dataset_folders[0]

print(f"Raw data folder: {RAW_DATA_DIR}")

Raw data folder: /kaggle/input/datasets


### Create output folders

In [11]:
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORTS_DIR = OUTPUT_DIR / "exports"

for folder in [CLEANED_DIR, REPORTS_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Cleaned documents: {CLEANED_DIR}")
print(f"Reports          : {REPORTS_DIR}")
print(f"Exports          : {EXPORTS_DIR}")

Cleaned documents: /kaggle/working/tax_llm/cleaned_documents
Reports          : /kaggle/working/tax_llm/reports
Exports          : /kaggle/working/tax_llm/exports


## Section 4: Discover Source Files

This section searches the selected dataset folder for supported tax-document files.

In [12]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = sorted(
    path
    for path in RAW_DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Source files found: {len(source_files)}")

if source_files:
    for path in source_files[:20]:
        print(f"- {path.relative_to(RAW_DATA_DIR)}")

    if len(source_files) > 20:
        print(f"... and {len(source_files) - 20} more files")
else:
    print("No supported files found. Check RAW_DATA_DIR and the dataset contents.")

Source files found: 58
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4163.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4164.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedul

## Section 5: Create the Document Inventory

This section creates a table containing basic information about every discovered source file.

In [13]:
import hashlib
import mimetypes

import pandas as pd
from tqdm.auto import tqdm


inventory_records = []

for file_path in tqdm(source_files, desc="Creating inventory"):
    relative_path = file_path.relative_to(RAW_DATA_DIR)
    file_stats = file_path.stat()

    # Create a stable ID from the relative file path
    document_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:16]

    inventory_records.append(
        {
            "document_id": document_id,
            "file_name": file_path.name,
            "relative_path": str(relative_path),
            "file_extension": file_path.suffix.lower(),
            "mime_type": mimetypes.guess_type(file_path.name)[0],
            "file_size_bytes": file_stats.st_size,
            "file_size_mb": round(file_stats.st_size / (1024 ** 2), 3),
        }
    )

document_inventory = pd.DataFrame(inventory_records)

print(f"Documents inventoried: {len(document_inventory)}")
display(document_inventory.head())

Creating inventory:   0%|          | 0/58 [00:00<?, ?it/s]

Documents inventoried: 58


,document_id,file_name,relative_path,file_extension,mime_type,file_size_bytes,file_size_mb
0,d304db736face09f,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,334608,0.319
1,afade3aec3a942e0,f1065x--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,168645,0.161
2,d1956ddbb121d7f5,p4163.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,1415699,1.350
3,caa77758d1eecb37,p4164.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,6909012,6.589
4,e54829d5a97de257,i1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,820966,0.783


## Review the inventory

In [14]:
print("Files by extension:")
display(
    document_inventory["file_extension"]
    .value_counts()
    .rename_axis("file_extension")
    .reset_index(name="file_count")
)

print(f"Total dataset size: {document_inventory['file_size_mb'].sum():.2f} MB")
print(f"Empty files found : {(document_inventory['file_size_bytes'] == 0).sum()}")

Files by extension:


,file_extension,file_count
0,.pdf,56
1,.csv,2


Total dataset size: 29.74 MB
Empty files found : 0


## save the inventory

In [15]:
inventory_path = REPORTS_DIR / "document_inventory.csv"

document_inventory.to_csv(inventory_path, index=False)

print(f"Inventory saved: {inventory_path}")

Inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 6: Detect Form 1065 and Form 1120 Documents

This section identifies whether each document relates to Form 1065, Form 1120, both forms, or neither form.

In [16]:
def get_identification_text(file_path, max_pdf_pages=3):
    """Read a small amount of text for form detection."""

    extension = file_path.suffix.lower()

    try:
        if extension == ".pdf":
            document = fitz.open(file_path)

            text = " ".join(
                document[page_number].get_text()
                for page_number in range(min(max_pdf_pages, len(document)))
            )

            document.close()
            return text

        if extension in {".txt", ".csv", ".json", ".jsonl"}:
            return file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )[:50_000]

        if extension in {".html", ".htm"}:
            html = file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            return BeautifulSoup(html, "lxml").get_text(" ")[:50_000]

    except Exception:
        return ""

    return ""

## Detect the return type

In [17]:
FORM_1065_PATTERN = re.compile(
    r"\bform[\s_-]*1065(?!\d)",
    re.IGNORECASE
)

FORM_1120_PATTERN = re.compile(
    r"\bform[\s_-]*1120(?![\s_-]?[a-z]|\d)",
    re.IGNORECASE
)


def detect_return_type(file_path):
    relative_path = str(
        file_path.relative_to(RAW_DATA_DIR)
    ).replace("_", " ").replace("-", " ")

    document_text = get_identification_text(file_path)

    searchable_text = f"{relative_path} {document_text[:50_000]}"

    has_1065 = bool(FORM_1065_PATTERN.search(searchable_text))
    has_1120 = bool(FORM_1120_PATTERN.search(searchable_text))

    if has_1065 and has_1120:
        return "both"
    elif has_1065:
        return "1065"
    elif has_1120:
        return "1120"
    else:
        return "unknown"

## Update the inventory

In [18]:
file_lookup = {
    str(path.relative_to(RAW_DATA_DIR)): path
    for path in source_files
}

detected_return_types = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting return types"
):
    file_path = file_lookup[relative_path]
    detected_return_types.append(
        detect_return_type(file_path)
    )

document_inventory["return_type"] = detected_return_types

display(
    document_inventory[
        ["file_name", "file_extension", "return_type"]
    ].head(20)
)

Detecting return types:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,file_extension,return_type
0,f1065--2025.pdf,.pdf,1065
1,f1065x--2025.pdf,.pdf,1065
2,p4163.pdf,.pdf,unknown
3,p4164.pdf,.pdf,unknown
4,i1065--2025.pdf,.pdf,1065
5,i1065x--2025.pdf,.pdf,1065
6,p541--2025.pdf,.pdf,1065
7,i1065s23--2025.pdf,.pdf,1065
8,i1065sb2--2018.pdf,.pdf,1065
9,i1065sd--2025.pdf,.pdf,1065


## Review and save results

In [19]:
return_type_summary = (
    document_inventory["return_type"]
    .value_counts(dropna=False)
    .rename_axis("return_type")
    .reset_index(name="document_count")
)

display(return_type_summary)

unknown_count = (
    document_inventory["return_type"] == "unknown"
).sum()

both_count = (
    document_inventory["return_type"] == "both"
).sum()

print(f"Unknown documents     : {unknown_count}")
print(f"Documents mentioning both forms: {both_count}")

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,return_type,document_count
0,1120,23
1,1065,19
2,unknown,12
3,both,4


Unknown documents     : 12
Documents mentioning both forms: 4
Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 7: Detect the Tax Year

This section detects the tax year using the file name and the first pages of each document.

## Tax year Detrection Function

In [20]:
from collections import Counter
from datetime import datetime


MIN_TAX_YEAR = 1990
MAX_TAX_YEAR = datetime.now().year + 1

YEAR_PATTERN = rf"\b(?:{MIN_TAX_YEAR}|19[9][0-9]|20[0-9]{{2}})\b"


def detect_tax_year(file_path):
    file_name_text = file_path.stem.replace("_", " ").replace("-", " ")
    document_text = get_identification_text(file_path, max_pdf_pages=3)

    year_scores = Counter()
    year_sources = {}

    # Years found in the file name receive higher priority
    filename_years = re.findall(YEAR_PATTERN, file_name_text)

    for year in filename_years:
        year_number = int(year)

        if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
            year_scores[year_number] += 5
            year_sources.setdefault(year_number, set()).add("file_name")

    # Search for clear tax-year statements
    tax_year_patterns = [
        rf"\btax\s+year\s+({YEAR_PATTERN})",
        rf"\bcalendar\s+year\s+({YEAR_PATTERN})",
        rf"\b({YEAR_PATTERN})\s+instructions?\s+for\s+form\b",
        rf"\b({YEAR_PATTERN})\s+form\s+(?:1065|1120)\b",
        rf"\bform\s+(?:1065|1120)\s*\(?({YEAR_PATTERN})\)?",
    ]

    for pattern in tax_year_patterns:
        for match in re.findall(pattern, document_text, flags=re.IGNORECASE):
            # Nested regex groups may return tuples
            if isinstance(match, tuple):
                match = next(
                    value for value in match
                    if value and re.fullmatch(r"\d{4}", value)
                )

            year_number = int(match)

            if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
                year_scores[year_number] += 3
                year_sources.setdefault(year_number, set()).add("document_text")

    if not year_scores:
        return {
            "tax_year": None,
            "tax_year_source": "not_detected",
            "tax_year_status": "review",
        }

    ranked_years = year_scores.most_common()
    best_year, best_score = ranked_years[0]

    # Flag equal scores for manual review
    tied_years = [
        year for year, score in ranked_years
        if score == best_score
    ]

    status = "detected" if len(tied_years) == 1 else "ambiguous"

    return {
        "tax_year": best_year,
        "tax_year_source": ", ".join(
            sorted(year_sources.get(best_year, {"unknown"}))
        ),
        "tax_year_status": status,
    }

In [21]:
# Update the inventory

tax_year_results = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting tax years"
):
    file_path = file_lookup[relative_path]
    tax_year_results.append(detect_tax_year(file_path))


tax_year_table = pd.DataFrame(tax_year_results)

document_inventory["tax_year"] = pd.array(
    tax_year_table["tax_year"],
    dtype="Int64"
)

document_inventory["tax_year_source"] = (
    tax_year_table["tax_year_source"]
)

document_inventory["tax_year_status"] = (
    tax_year_table["tax_year_status"]
)

display(
    document_inventory[
        [
            "file_name",
            "return_type",
            "tax_year",
            "tax_year_source",
            "tax_year_status",
        ]
    ].head(20)
)

Detecting tax years:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,return_type,tax_year,tax_year_source,tax_year_status
0,f1065--2025.pdf,1065,2025,"document_text, file_name",detected
1,f1065x--2025.pdf,1065,2025,file_name,detected
2,p4163.pdf,unknown,<NA>,not_detected,review
3,p4164.pdf,unknown,<NA>,not_detected,review
4,i1065--2025.pdf,1065,2025,"document_text, file_name",detected
5,i1065x--2025.pdf,1065,2025,file_name,detected
6,p541--2025.pdf,1065,2025,file_name,detected
7,i1065s23--2025.pdf,1065,2025,"document_text, file_name",detected
8,i1065sb2--2018.pdf,1065,2018,file_name,detected
9,i1065sd--2025.pdf,1065,2025,file_name,detected


In [22]:
# Review and save the results

tax_year_summary = (
    document_inventory
    .groupby(["tax_year", "return_type"], dropna=False)
    .size()
    .reset_index(name="document_count")
    .sort_values(["tax_year", "return_type"])
)

display(tax_year_summary)

review_documents = document_inventory[
    document_inventory["tax_year_status"] != "detected"
]

print(f"Detected tax years : {(document_inventory['tax_year_status'] == 'detected').sum()}")
print(f"Needs review       : {len(review_documents)}")

if not review_documents.empty:
    display(
        review_documents[
            ["file_name", "return_type", "tax_year", "tax_year_status"]
        ].head(20)
    )

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,tax_year,return_type,document_count
0,2011,1120,2
1,2014,1065,1
2,2015,1120,1
3,2016,1120,2
4,2016,unknown,1
5,2018,1065,2
6,2018,1120,4
7,2018,both,1
8,2019,1065,1
9,2019,1120,1


Detected tax years : 54
Needs review       : 4


,file_name,return_type,tax_year,tax_year_status
2,p4163.pdf,unknown,<NA>,review
3,p4164.pdf,unknown,<NA>,review
38,p4163.pdf,unknown,<NA>,review
39,p4164.pdf,unknown,<NA>,review


Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 8: Extract Text from Digital PDFs

This section extracts text from searchable PDF files and identifies PDFs that may require OCR.

## Create extraction folder

In [23]:
PDF_TEXT_DIR = CLEANED_DIR / "pdf_text"
PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

MIN_PAGE_CHARACTERS = 50
MIN_DOCUMENT_CHARACTERS = 200

print(f"PDF text folder: {PDF_TEXT_DIR}")

PDF text folder: /kaggle/working/tax_llm/cleaned_documents/pdf_text


## Define the extraction function

In [24]:
def extract_pdf_text(file_path, document_id):
    output_path = PDF_TEXT_DIR / f"{document_id}.txt"

    try:
        page_texts = []

        with fitz.open(file_path) as pdf_document:
            page_count = len(pdf_document)

            for page_number, page in enumerate(pdf_document, start=1):
                text = page.get_text("text").strip()

                page_texts.append(
                    f"\n--- Page {page_number} ---\n{text}"
                )

        full_text = "\n".join(page_texts).strip()
        character_count = len(full_text)

        pages_with_text = sum(
            len(page_text.strip()) >= MIN_PAGE_CHARACTERS
            for page_text in page_texts
        )

        text_page_ratio = (
            pages_with_text / page_count
            if page_count > 0
            else 0
        )

        if (
            character_count >= MIN_DOCUMENT_CHARACTERS
            and text_page_ratio >= 0.50
        ):
            extraction_status = "digital_text_extracted"
        else:
            extraction_status = "needs_ocr"

        output_path.write_text(
            full_text,
            encoding="utf-8"
        )

        return {
            "page_count": page_count,
            "character_count": character_count,
            "pages_with_text": pages_with_text,
            "text_page_ratio": round(text_page_ratio, 3),
            "extraction_status": extraction_status,
            "text_path": str(output_path.relative_to(OUTPUT_DIR)),
            "extraction_error": None,
        }

    except Exception as error:
        return {
            "page_count": None,
            "character_count": 0,
            "pages_with_text": 0,
            "text_page_ratio": 0,
            "extraction_status": "failed",
            "text_path": None,
            "extraction_error": str(error),
        }

## Extract all PDFs

In [25]:
extraction_results = []

pdf_inventory = document_inventory[
    document_inventory["file_extension"] == ".pdf"
]

for row in tqdm(
    pdf_inventory.itertuples(index=False),
    total=len(pdf_inventory),
    desc="Extracting PDF text"
):
    file_path = file_lookup[row.relative_path]

    result = extract_pdf_text(
        file_path=file_path,
        document_id=row.document_id,
    )

    result["document_id"] = row.document_id
    extraction_results.append(result)

pdf_extraction_table = pd.DataFrame(extraction_results)

print(f"PDFs processed: {len(pdf_extraction_table)}")
display(pdf_extraction_table.head())

Extracting PDF text:   0%|          | 0/56 [00:00<?, ?it/s]

PDFs processed: 56


,page_count,character_count,pages_with_text,text_page_ratio,extraction_status,text_path,extraction_error,document_id
0,6,25687,6,1.000,digital_text_extracted,cleaned_documents/pdf_text/d304db736face09f.txt,None,d304db736face09f
1,4,9804,4,1.000,digital_text_extracted,cleaned_documents/pdf_text/afade3aec3a942e0.txt,None,afade3aec3a942e0
2,96,226522,96,1.000,digital_text_extracted,cleaned_documents/pdf_text/d1956ddbb121d7f5.txt,None,d1956ddbb121d7f5
3,282,474824,280,0.993,digital_text_extracted,cleaned_documents/pdf_text/caa77758d1eecb37.txt,None,caa77758d1eecb37
4,70,458426,70,1.000,digital_text_extracted,cleaned_documents/pdf_text/e54829d5a97de257.txt,None,e54829d5a97de257


## Add results to the inventory

In [26]:
extraction_columns = [
    "document_id",
    "page_count",
    "character_count",
    "pages_with_text",
    "text_page_ratio",
    "extraction_status",
    "text_path",
    "extraction_error",
]

document_inventory = document_inventory.drop(
    columns=extraction_columns[1:],
    errors="ignore",
)

document_inventory = document_inventory.merge(
    pdf_extraction_table[extraction_columns],
    on="document_id",
    how="left",
)

document_inventory["extraction_status"] = (
    document_inventory["extraction_status"]
    .fillna("not_processed")
)

display(
    document_inventory[
        [
            "file_name",
            "page_count",
            "character_count",
            "text_page_ratio",
            "extraction_status",
        ]
    ].head(20)
)

,file_name,page_count,character_count,text_page_ratio,extraction_status
0,f1065--2025.pdf,6.0,25687.0,1.000,digital_text_extracted
1,f1065x--2025.pdf,4.0,9804.0,1.000,digital_text_extracted
2,p4163.pdf,96.0,226522.0,1.000,digital_text_extracted
3,p4164.pdf,282.0,474824.0,0.993,digital_text_extracted
4,i1065--2025.pdf,70.0,458426.0,1.000,digital_text_extracted
5,i1065x--2025.pdf,12.0,64008.0,1.000,digital_text_extracted
6,p541--2025.pdf,33.0,162331.0,1.000,digital_text_extracted
7,i1065s23--2025.pdf,47.0,307743.0,1.000,digital_text_extracted
8,i1065sb2--2018.pdf,2.0,7694.0,1.000,digital_text_extracted
9,i1065sd--2025.pdf,6.0,35125.0,1.000,digital_text_extracted


## Review and save

In [27]:
display(
    document_inventory["extraction_status"]
    .value_counts(dropna=False)
    .rename_axis("extraction_status")
    .reset_index(name="document_count")
)

needs_ocr = document_inventory[
    document_inventory["extraction_status"] == "needs_ocr"
]

failed_extractions = document_inventory[
    document_inventory["extraction_status"] == "failed"
]

print(f"PDFs requiring OCR : {len(needs_ocr)}")
print(f"Failed extractions : {len(failed_extractions)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

pdf_extraction_table.to_csv(
    REPORTS_DIR / "pdf_extraction_report.csv",
    index=False,
)

print("Extraction results saved successfully.")

,extraction_status,document_count
0,digital_text_extracted,56
1,not_processed,2


PDFs requiring OCR : 0
Failed extractions : 0
Extraction results saved successfully.


## Section 9: Apply OCR to Scanned PDFs

This section uses OCR to extract text from scanned PDFs that could not be read digitally.

OCR uses the CPU and may take longer than normal PDF extraction.

## Configure OCR

In [28]:
OCR_TEXT_DIR = CLEANED_DIR / "ocr_text"
OCR_TEXT_DIR.mkdir(parents=True, exist_ok=True)

OCR_DPI = 200
OCR_CONFIG = "--oem 3 --psm 3"

ocr_candidates = document_inventory[
    document_inventory["extraction_status"] == "needs_ocr"
].copy()

print(f"PDFs requiring OCR: {len(ocr_candidates)}")
print(f"OCR output folder : {OCR_TEXT_DIR}")

PDFs requiring OCR: 0
OCR output folder : /kaggle/working/tax_llm/cleaned_documents/ocr_text


## Define the OCR function

In [29]:
def extract_pdf_with_ocr(file_path, document_id):
    output_path = OCR_TEXT_DIR / f"{document_id}.txt"

    try:
        page_texts = []
        ocr_page_count = 0
        scale = OCR_DPI / 72

        with fitz.open(file_path) as pdf_document:
            page_count = len(pdf_document)

            for page_number, page in enumerate(pdf_document, start=1):
                embedded_text = page.get_text("text").strip()

                # Keep usable digital text in mixed PDFs
                if len(embedded_text) >= MIN_PAGE_CHARACTERS:
                    page_text = embedded_text
                else:
                    pixmap = page.get_pixmap(
                        matrix=fitz.Matrix(scale, scale),
                        colorspace=fitz.csRGB,
                        alpha=False,
                    )

                    image = Image.frombytes(
                        "RGB",
                        (pixmap.width, pixmap.height),
                        pixmap.samples,
                    )

                    page_text = pytesseract.image_to_string(
                        image,
                        lang="eng",
                        config=OCR_CONFIG,
                    ).strip()

                    image.close()
                    ocr_page_count += 1

                page_texts.append(
                    f"--- Page {page_number} ---\n{page_text}"
                )

        full_text = "\n\n".join(page_texts).strip()
        character_count = len(full_text)

        if character_count >= MIN_DOCUMENT_CHARACTERS:
            status = "ocr_extracted"
        else:
            status = "ocr_low_text"

        output_path.write_text(full_text, encoding="utf-8")

        return {
            "document_id": document_id,
            "ocr_status": status,
            "ocr_page_count": ocr_page_count,
            "ocr_character_count": character_count,
            "ocr_text_path": str(output_path.relative_to(OUTPUT_DIR)),
            "ocr_error": None,
        }

    except Exception as error:
        return {
            "document_id": document_id,
            "ocr_status": "ocr_failed",
            "ocr_page_count": 0,
            "ocr_character_count": 0,
            "ocr_text_path": None,
            "ocr_error": str(error),
        }

## Test OCR on one PDF

In [30]:
if ocr_candidates.empty:
    print("No PDFs require OCR.")
else:
    test_row = ocr_candidates.iloc[0]
    test_file = file_lookup[test_row["relative_path"]]

    test_result = extract_pdf_with_ocr(
        test_file,
        test_row["document_id"],
    )

    print(test_result)

No PDFs require OCR.


In [31]:
# Process all OCR candidates

ocr_results = []

for row in tqdm(
    ocr_candidates.itertuples(index=False),
    total=len(ocr_candidates),
    desc="Applying OCR",
):
    file_path = file_lookup[row.relative_path]

    result = extract_pdf_with_ocr(
        file_path=file_path,
        document_id=row.document_id,
    )

    ocr_results.append(result)

ocr_table = pd.DataFrame(ocr_results)

print(f"PDFs processed with OCR: {len(ocr_table)}")

if not ocr_table.empty:
    display(ocr_table.head())

Applying OCR: 0it [00:00, ?it/s]

PDFs processed with OCR: 0


In [32]:
## Update and save the inventory

if not ocr_table.empty:
    ocr_columns = [
        "ocr_status",
        "ocr_page_count",
        "ocr_character_count",
        "ocr_text_path",
        "ocr_error",
    ]

    for column in ocr_columns:
        if column not in document_inventory.columns:
            document_inventory[column] = None

    ocr_lookup = ocr_table.set_index("document_id")

    for document_id in ocr_lookup.index:
        mask = document_inventory["document_id"] == document_id

        for column in ocr_columns:
            document_inventory.loc[mask, column] = (
                ocr_lookup.loc[document_id, column]
            )

        status = ocr_lookup.loc[document_id, "ocr_status"]

        document_inventory.loc[mask, "extraction_status"] = status

        if status in {"ocr_extracted", "ocr_low_text"}:
            document_inventory.loc[mask, "text_path"] = (
                ocr_lookup.loc[document_id, "ocr_text_path"]
            )

    display(
        document_inventory["extraction_status"]
        .value_counts(dropna=False)
        .rename_axis("extraction_status")
        .reset_index(name="document_count")
    )

    ocr_table.to_csv(
        REPORTS_DIR / "ocr_extraction_report.csv",
        index=False,
    )

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

print("OCR results saved successfully.")

OCR results saved successfully.


## Section 10: Extract Text from Non-PDF Files

This section extracts text from TXT, HTML, CSV, JSON, and JSONL files.

In [33]:
NON_PDF_TEXT_DIR = CLEANED_DIR / "non_pdf_text"
NON_PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

NON_PDF_EXTENSIONS = {
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

non_pdf_inventory = document_inventory[
    document_inventory["file_extension"].isin(NON_PDF_EXTENSIONS)
].copy()

print(f"Non-PDF files found: {len(non_pdf_inventory)}")
print(f"Output folder      : {NON_PDF_TEXT_DIR}")

Non-PDF files found: 2
Output folder      : /kaggle/working/tax_llm/cleaned_documents/non_pdf_text


## Configure extraction

In [34]:
NON_PDF_TEXT_DIR = CLEANED_DIR / "non_pdf_text"
NON_PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

NON_PDF_EXTENSIONS = {
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

non_pdf_inventory = document_inventory[
    document_inventory["file_extension"].isin(NON_PDF_EXTENSIONS)
].copy()

print(f"Non-PDF files found: {len(non_pdf_inventory)}")
print(f"Output folder      : {NON_PDF_TEXT_DIR}")

Non-PDF files found: 2
Output folder      : /kaggle/working/tax_llm/cleaned_documents/non_pdf_text


## Define the extraction function

In [35]:
def extract_non_pdf_text(file_path, document_id):
    output_path = NON_PDF_TEXT_DIR / f"{document_id}.txt"

    try:
        raw_text = file_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        if file_path.suffix.lower() in {".html", ".htm"}:
            text = BeautifulSoup(
                raw_text,
                "lxml",
            ).get_text(separator="\n")

        else:
            text = raw_text

        # Remove null characters and excessive blank lines
        text = text.replace("\x00", "")
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        character_count = len(text)

        status = (
            "text_extracted"
            if character_count >= MIN_DOCUMENT_CHARACTERS
            else "low_text"
        )

        output_path.write_text(text, encoding="utf-8")

        return {
            "document_id": document_id,
            "non_pdf_status": status,
            "non_pdf_character_count": character_count,
            "non_pdf_text_path": str(
                output_path.relative_to(OUTPUT_DIR)
            ),
            "non_pdf_error": None,
        }

    except Exception as error:
        return {
            "document_id": document_id,
            "non_pdf_status": "failed",
            "non_pdf_character_count": 0,
            "non_pdf_text_path": None,
            "non_pdf_error": str(error),
        }

## Process the files

In [36]:
non_pdf_results = []

for row in tqdm(
    non_pdf_inventory.itertuples(index=False),
    total=len(non_pdf_inventory),
    desc="Extracting non-PDF text",
):
    file_path = file_lookup[row.relative_path]

    result = extract_non_pdf_text(
        file_path=file_path,
        document_id=row.document_id,
    )

    non_pdf_results.append(result)

non_pdf_table = pd.DataFrame(non_pdf_results)

print(f"Non-PDF files processed: {len(non_pdf_table)}")

if not non_pdf_table.empty:
    display(non_pdf_table.head())

Extracting non-PDF text:   0%|          | 0/2 [00:00<?, ?it/s]

Non-PDF files processed: 2


,document_id,non_pdf_status,non_pdf_character_count,non_pdf_text_path,non_pdf_error
0,9a21200b067fa27b,text_extracted,2418,cleaned_documents/non_pdf_text/9a21200b067fa27...,None
1,243c634d6655d5f1,text_extracted,2577,cleaned_documents/non_pdf_text/243c634d6655d5f...,None


## Update the inventory

In [37]:
if not non_pdf_table.empty:
    result_lookup = non_pdf_table.set_index("document_id")

    for document_id, result in result_lookup.iterrows():
        mask = document_inventory["document_id"] == document_id

        document_inventory.loc[
            mask, "character_count"
        ] = result["non_pdf_character_count"]

        document_inventory.loc[
            mask, "extraction_status"
        ] = result["non_pdf_status"]

        document_inventory.loc[
            mask, "text_path"
        ] = result["non_pdf_text_path"]

        document_inventory.loc[
            mask, "extraction_error"
        ] = result["non_pdf_error"]

## Review and save

In [38]:
display(
    document_inventory["extraction_status"]
    .value_counts(dropna=False)
    .rename_axis("extraction_status")
    .reset_index(name="document_count")
)

failed_non_pdf = non_pdf_table[
    non_pdf_table["non_pdf_status"] == "failed"
] if not non_pdf_table.empty else pd.DataFrame()

print(f"Failed non-PDF extractions: {len(failed_non_pdf)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

if not non_pdf_table.empty:
    non_pdf_table.to_csv(
        REPORTS_DIR / "non_pdf_extraction_report.csv",
        index=False,
    )

print("Non-PDF extraction results saved successfully.")

,extraction_status,document_count
0,digital_text_extracted,56
1,text_extracted,2


Failed non-PDF extractions: 0
Non-PDF extraction results saved successfully.


## Section 11: Clean and Normalize Extracted Text


This section removes unwanted characters and normalizes spacing while preserving page boundaries and tax-document structure.

## Create the cleaned-text folder

In [39]:
import unicodedata

NORMALIZED_TEXT_DIR = CLEANED_DIR / "normalized_text"
NORMALIZED_TEXT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Normalized text folder: {NORMALIZED_TEXT_DIR}")

Normalized text folder: /kaggle/working/tax_llm/cleaned_documents/normalized_text


## Define the cleaning function

In [40]:
def clean_extracted_text(text):
    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove null bytes and control characters
    text = text.replace("\x00", "")
    text = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Join words split across lines by PDF extraction
    text = re.sub(
        r"(?<=[a-z])-\n(?=[a-z])",
        "",
        text,
    )

    # Replace repeated spaces and tabs without removing line breaks
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    # Limit excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Clean all extracted documents

In [41]:
cleaning_results = []

documents_with_text = document_inventory[
    document_inventory["text_path"].notna()
].copy()

for row in tqdm(
    documents_with_text.itertuples(index=False),
    total=len(documents_with_text),
    desc="Cleaning extracted text",
):
    try:
        source_text_path = OUTPUT_DIR / row.text_path
        normalized_path = NORMALIZED_TEXT_DIR / f"{row.document_id}.txt"

        raw_text = source_text_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        cleaned_text = clean_extracted_text(raw_text)
        normalized_path.write_text(cleaned_text, encoding="utf-8")

        cleaning_results.append(
            {
                "document_id": row.document_id,
                "raw_character_count": len(raw_text),
                "clean_character_count": len(cleaned_text),
                "clean_text_path": str(
                    normalized_path.relative_to(OUTPUT_DIR)
                ),
                "cleaning_status": (
                    "cleaned"
                    if len(cleaned_text) >= MIN_DOCUMENT_CHARACTERS
                    else "low_text"
                ),
                "cleaning_error": None,
            }
        )

    except Exception as error:
        cleaning_results.append(
            {
                "document_id": row.document_id,
                "raw_character_count": 0,
                "clean_character_count": 0,
                "clean_text_path": None,
                "cleaning_status": "failed",
                "cleaning_error": str(error),
            }
        )

cleaning_table = pd.DataFrame(cleaning_results)

print(f"Documents processed: {len(cleaning_table)}")

if not cleaning_table.empty:
    display(cleaning_table.head())

Cleaning extracted text:   0%|          | 0/58 [00:00<?, ?it/s]

Documents processed: 58


,document_id,raw_character_count,clean_character_count,clean_text_path,cleaning_status,cleaning_error
0,d304db736face09f,25687,24921,cleaned_documents/normalized_text/d304db736fac...,cleaned,None
1,afade3aec3a942e0,9804,9629,cleaned_documents/normalized_text/afade3aec3a9...,cleaned,None
2,d1956ddbb121d7f5,226522,221609,cleaned_documents/normalized_text/d1956ddbb121...,cleaned,None
3,caa77758d1eecb37,474824,462892,cleaned_documents/normalized_text/caa77758d1ee...,cleaned,None
4,e54829d5a97de257,458426,452312,cleaned_documents/normalized_text/e54829d5a97d...,cleaned,None


## Update the inventory

In [42]:
cleaning_columns = [
    "document_id",
    "raw_character_count",
    "clean_character_count",
    "clean_text_path",
    "cleaning_status",
    "cleaning_error",
]

document_inventory = document_inventory.drop(
    columns=cleaning_columns[1:],
    errors="ignore",
)

if not cleaning_table.empty:
    document_inventory = document_inventory.merge(
        cleaning_table[cleaning_columns],
        on="document_id",
        how="left",
    )

document_inventory["cleaning_status"] = (
    document_inventory["cleaning_status"]
    .fillna("not_processed")
)

## Review and save

In [43]:
display(
    document_inventory["cleaning_status"]
    .value_counts(dropna=False)
    .rename_axis("cleaning_status")
    .reset_index(name="document_count")
)

failed_cleaning = document_inventory[
    document_inventory["cleaning_status"] == "failed"
]

low_text_cleaning = document_inventory[
    document_inventory["cleaning_status"] == "low_text"
]

print(f"Cleaning failures : {len(failed_cleaning)}")
print(f"Low-text documents: {len(low_text_cleaning)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

cleaning_table.to_csv(
    REPORTS_DIR / "text_cleaning_report.csv",
    index=False,
)

print("Cleaned text and reports saved successfully.")

,cleaning_status,document_count
0,cleaned,58


Cleaning failures : 0
Low-text documents: 0
Cleaned text and reports saved successfully.


## Section 12: Detect Exact and Near-Duplicate Documents

This section detects identical and highly similar documents to prevent duplicate training data.

In [44]:
def normalize_for_duplicate_check(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


duplicate_records = []
document_texts = {}

cleaned_documents = document_inventory[
    document_inventory["clean_text_path"].notna()
].copy()

for row in tqdm(
    cleaned_documents.itertuples(index=False),
    total=len(cleaned_documents),
    desc="Calculating document hashes",
):
    try:
        text_path = OUTPUT_DIR / row.clean_text_path
        text = text_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        normalized_text = normalize_for_duplicate_check(text)
        document_texts[row.document_id] = normalized_text

        text_hash = hashlib.sha256(
            normalized_text.encode("utf-8")
        ).hexdigest()

        duplicate_records.append(
            {
                "document_id": row.document_id,
                "text_hash": text_hash,
                "duplicate_error": None,
            }
        )

    except Exception as error:
        duplicate_records.append(
            {
                "document_id": row.document_id,
                "text_hash": None,
                "duplicate_error": str(error),
            }
        )

hash_table = pd.DataFrame(duplicate_records)

print(f"Documents hashed: {len(hash_table)}")
display(hash_table.head())

Calculating document hashes:   0%|          | 0/58 [00:00<?, ?it/s]

Documents hashed: 58


,document_id,text_hash,duplicate_error
0,d304db736face09f,65e1f8c96ccd034972267865827673884fd05e005dbeb4...,None
1,afade3aec3a942e0,a7295a055533aa0a7829be5fd755e18a92f3c9697e0dae...,None
2,d1956ddbb121d7f5,e93ae09b1cb015e0bcdc1849665aff703943fa1a95c9d4...,None
3,caa77758d1eecb37,1108a2e749f938675c7f57ff13af43712826e72f0565af...,None
4,e54829d5a97de257,65ae7eecc1409dc3f89dc381b6db6f21e2d1ac4db2432e...,None


## Detect exact duplicates

In [45]:
valid_hashes = hash_table[
    hash_table["text_hash"].notna()
].copy()

valid_hashes["exact_duplicate_rank"] = (
    valid_hashes.groupby("text_hash").cumcount()
)

valid_hashes["exact_duplicate"] = (
    valid_hashes["exact_duplicate_rank"] > 0
)

canonical_lookup = (
    valid_hashes.groupby("text_hash")["document_id"]
    .first()
    .to_dict()
)

valid_hashes["duplicate_of"] = valid_hashes.apply(
    lambda row: (
        canonical_lookup[row["text_hash"]]
        if row["exact_duplicate"]
        else None
    ),
    axis=1,
)

exact_duplicate_count = valid_hashes["exact_duplicate"].sum()

print(f"Exact duplicates found: {exact_duplicate_count}")

display(
    valid_hashes[
        valid_hashes["exact_duplicate"]
    ][
        ["document_id", "duplicate_of", "text_hash"]
    ].head(20)
)

Exact duplicates found: 2


,document_id,duplicate_of,text_hash
38,687f60f4af63f749,d1956ddbb121d7f5,e93ae09b1cb015e0bcdc1849665aff703943fa1a95c9d4...
39,f44c63cc48a3be4d,caa77758d1eecb37,1108a2e749f938675c7f57ff13af43712826e72f0565af...


## Detect near duplicates

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


# Compare only canonical documents with enough text
canonical_documents = valid_hashes[
    ~valid_hashes["exact_duplicate"]
]["document_id"].tolist()

candidate_ids = [
    document_id
    for document_id in canonical_documents
    if len(document_texts.get(document_id, "")) >= MIN_DOCUMENT_CHARACTERS
]

candidate_texts = [
    document_texts[document_id]
    for document_id in candidate_ids
]

NEAR_DUPLICATE_THRESHOLD = 0.97
near_duplicate_records = []

if len(candidate_texts) >= 2:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        analyzer="word",
        ngram_range=(1, 2),
        min_df=1,
        max_features=100_000,
        sublinear_tf=True,
    )

    text_vectors = vectorizer.fit_transform(candidate_texts)

    neighbor_count = min(5, len(candidate_ids))

    neighbor_model = NearestNeighbors(
        n_neighbors=neighbor_count,
        metric="cosine",
        algorithm="brute",
    )

    neighbor_model.fit(text_vectors)

    distances, indices = neighbor_model.kneighbors(text_vectors)

    compared_pairs = set()

    for source_index, document_id in enumerate(candidate_ids):
        for distance, target_index in zip(
            distances[source_index],
            indices[source_index],
        ):
            target_id = candidate_ids[target_index]

            if document_id == target_id:
                continue

            pair = tuple(sorted([document_id, target_id]))

            if pair in compared_pairs:
                continue

            compared_pairs.add(pair)
            similarity = 1 - float(distance)

            if similarity >= NEAR_DUPLICATE_THRESHOLD:
                near_duplicate_records.append(
                    {
                        "document_id_1": pair[0],
                        "document_id_2": pair[1],
                        "similarity_score": round(similarity, 4),
                        "review_status": "needs_review",
                    }
                )

near_duplicate_table = pd.DataFrame(
    near_duplicate_records,
    columns=[
        "document_id_1",
        "document_id_2",
        "similarity_score",
        "review_status",
    ],
)

print(f"Near-duplicate pairs found: {len(near_duplicate_table)}")

if not near_duplicate_table.empty:
    display(
        near_duplicate_table.sort_values(
            "similarity_score",
            ascending=False,
        ).head(20)
    )

Near-duplicate pairs found: 0


## Update the inventory

In [47]:
duplicate_columns = [
    "text_hash",
    "exact_duplicate",
    "duplicate_of",
]

document_inventory = document_inventory.drop(
    columns=duplicate_columns,
    errors="ignore",
)

document_inventory = document_inventory.merge(
    valid_hashes[
        [
            "document_id",
            "text_hash",
            "exact_duplicate",
            "duplicate_of",
        ]
    ],
    on="document_id",
    how="left",
)

document_inventory["exact_duplicate"] = (
    document_inventory["exact_duplicate"]
    .fillna(False)
    .astype(bool)
)

document_inventory["duplicate_status"] = "unique"

document_inventory.loc[
    document_inventory["exact_duplicate"],
    "duplicate_status",
] = "exact_duplicate"

# Near duplicates require manual review before removal
if not near_duplicate_table.empty:
    near_duplicate_ids = set(
        near_duplicate_table["document_id_1"]
    ) | set(
        near_duplicate_table["document_id_2"]
    )

    near_duplicate_mask = (
        document_inventory["document_id"].isin(near_duplicate_ids)
        & ~document_inventory["exact_duplicate"]
    )

    document_inventory.loc[
        near_duplicate_mask,
        "duplicate_status",
    ] = "possible_near_duplicate"

## Review and save

In [48]:
display(
    document_inventory["duplicate_status"]
    .value_counts(dropna=False)
    .rename_axis("duplicate_status")
    .reset_index(name="document_count")
)

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

valid_hashes.to_csv(
    REPORTS_DIR / "exact_duplicate_report.csv",
    index=False,
)

near_duplicate_table.to_csv(
    REPORTS_DIR / "near_duplicate_report.csv",
    index=False,
)

print("Duplicate reports saved successfully.")

,duplicate_status,document_count
0,unique,56
1,exact_duplicate,2


Duplicate reports saved successfully.


## Section 13: Validate Document Quality

This section calculates text-quality metrics, assigns a quality score, and identifies documents that require review or exclusion.

### Configure quality thresholds

In [49]:
MIN_QUALITY_CHARACTERS = 200
MIN_QUALITY_WORDS = 50
MIN_ALPHABETIC_RATIO = 0.45
MAX_REPLACEMENT_RATIO = 0.01

QUALITY_SCORE_REVIEW_THRESHOLD = 60
QUALITY_SCORE_GOOD_THRESHOLD = 80

### Define the quality-check function

In [50]:
def calculate_text_quality(text):
    character_count = len(text)
    word_count = len(re.findall(r"\b\w+\b", text))

    alphabetic_count = sum(
        character.isalpha()
        for character in text
    )

    printable_count = sum(
        character.isprintable() or character in "\n\t"
        for character in text
    )

    replacement_count = text.count("\ufffd")

    alphabetic_ratio = (
        alphabetic_count / character_count
        if character_count
        else 0
    )

    printable_ratio = (
        printable_count / character_count
        if character_count
        else 0
    )

    replacement_ratio = (
        replacement_count / character_count
        if character_count
        else 0
    )

    return {
        "quality_character_count": character_count,
        "quality_word_count": word_count,
        "alphabetic_ratio": round(alphabetic_ratio, 4),
        "printable_ratio": round(printable_ratio, 4),
        "replacement_character_ratio": round(
            replacement_ratio,
            4,
        ),
    }

### Score each document

In [51]:
def score_document_quality(inventory_row, text):
    metrics = calculate_text_quality(text)

    score = 100
    flags = []

    character_count = metrics["quality_character_count"]
    word_count = metrics["quality_word_count"]
    alphabetic_ratio = metrics["alphabetic_ratio"]
    printable_ratio = metrics["printable_ratio"]
    replacement_ratio = metrics["replacement_character_ratio"]

    if character_count < MIN_QUALITY_CHARACTERS:
        score -= 40
        flags.append("insufficient_text")
    elif character_count < 1_000:
        score -= 15
        flags.append("short_document")

    if word_count < MIN_QUALITY_WORDS:
        score -= 20
        flags.append("low_word_count")

    if alphabetic_ratio < MIN_ALPHABETIC_RATIO:
        score -= 20
        flags.append("low_alphabetic_ratio")

    if printable_ratio < 0.95:
        score -= 15
        flags.append("non_printable_characters")

    if replacement_ratio > MAX_REPLACEMENT_RATIO:
        score -= 20
        flags.append("encoding_or_ocr_noise")

    extraction_status = str(
        inventory_row.get("extraction_status", "")
    )

    cleaning_status = str(
        inventory_row.get("cleaning_status", "")
    )

    duplicate_status = str(
        inventory_row.get("duplicate_status", "")
    )

    return_type = str(
        inventory_row.get("return_type", "")
    )

    tax_year_status = str(
        inventory_row.get("tax_year_status", "")
    )

    if extraction_status in {
        "failed",
        "ocr_failed",
        "ocr_low_text",
        "low_text",
    }:
        score -= 25
        flags.append(f"extraction_{extraction_status}")

    if cleaning_status == "failed":
        score -= 30
        flags.append("cleaning_failed")

    if return_type in {"unknown", "both", "", "nan"}:
        score -= 10
        flags.append("return_type_review")

    if tax_year_status != "detected":
        score -= 10
        flags.append("tax_year_review")

    if duplicate_status == "exact_duplicate":
        score = 0
        flags.append("exact_duplicate")

    elif duplicate_status == "possible_near_duplicate":
        score -= 5
        flags.append("possible_near_duplicate")

    score = max(0, min(100, score))

    if score >= QUALITY_SCORE_GOOD_THRESHOLD:
        quality_grade = "good"
    elif score >= QUALITY_SCORE_REVIEW_THRESHOLD:
        quality_grade = "acceptable"
    else:
        quality_grade = "poor"

    mandatory_exclusion_flags = {
        "insufficient_text",
        "cleaning_failed",
        "exact_duplicate",
        "extraction_failed",
        "extraction_ocr_failed",
    }

    if mandatory_exclusion_flags.intersection(flags):
        training_status = "exclude"
    elif quality_grade == "poor" or flags:
        training_status = "review"
    else:
        training_status = "eligible"

    return {
        **metrics,
        "quality_score": score,
        "quality_grade": quality_grade,
        "quality_flags": (
            ";".join(sorted(set(flags)))
            if flags
            else "none"
        ),
        "training_status": training_status,
    }

### Validate all cleaned documents

In [52]:
quality_results = []

for _, row in tqdm(
    document_inventory.iterrows(),
    total=len(document_inventory),
    desc="Validating document quality",
):
    document_id = row["document_id"]
    clean_text_path = row.get("clean_text_path")

    if pd.isna(clean_text_path):
        quality_results.append(
            {
                "document_id": document_id,
                "quality_character_count": 0,
                "quality_word_count": 0,
                "alphabetic_ratio": 0,
                "printable_ratio": 0,
                "replacement_character_ratio": 0,
                "quality_score": 0,
                "quality_grade": "poor",
                "quality_flags": "missing_clean_text",
                "training_status": "exclude",
                "quality_error": None,
            }
        )
        continue

    try:
        text_file = OUTPUT_DIR / clean_text_path

        cleaned_text = text_file.read_text(
            encoding="utf-8",
            errors="replace",
        )

        result = score_document_quality(
            inventory_row=row,
            text=cleaned_text,
        )

        result["document_id"] = document_id
        result["quality_error"] = None
        quality_results.append(result)

    except Exception as error:
        quality_results.append(
            {
                "document_id": document_id,
                "quality_character_count": 0,
                "quality_word_count": 0,
                "alphabetic_ratio": 0,
                "printable_ratio": 0,
                "replacement_character_ratio": 0,
                "quality_score": 0,
                "quality_grade": "poor",
                "quality_flags": "quality_check_failed",
                "training_status": "exclude",
                "quality_error": str(error),
            }
        )

quality_table = pd.DataFrame(quality_results)

print(f"Documents validated: {len(quality_table)}")
display(quality_table.head())

Validating document quality:   0%|          | 0/58 [00:00<?, ?it/s]

Documents validated: 58


,quality_character_count,quality_word_count,alphabetic_ratio,printable_ratio,replacement_character_ratio,quality_score,quality_grade,quality_flags,training_status,document_id,quality_error
0,24921,3447,0.6106,1.0000,0.0,100,good,none,eligible,d304db736face09f,None
1,9629,1475,0.6510,1.0000,0.0,100,good,none,eligible,afade3aec3a942e0,None
2,221609,33941,0.7027,1.0000,0.0,80,good,return_type_review;tax_year_review,review,d1956ddbb121d7f5,None
3,462892,71729,0.7091,0.9999,0.0,80,good,return_type_review;tax_year_review,review,caa77758d1eecb37,None
4,452312,73817,0.7732,1.0000,0.0,100,good,none,eligible,e54829d5a97de257,None


### Update the inventory

In [53]:
quality_columns = [
    "document_id",
    "quality_character_count",
    "quality_word_count",
    "alphabetic_ratio",
    "printable_ratio",
    "replacement_character_ratio",
    "quality_score",
    "quality_grade",
    "quality_flags",
    "training_status",
    "quality_error",
]

document_inventory = document_inventory.drop(
    columns=quality_columns[1:],
    errors="ignore",
)

document_inventory = document_inventory.merge(
    quality_table[quality_columns],
    on="document_id",
    how="left",
)

display(
    document_inventory[
        [
            "file_name",
            "return_type",
            "tax_year",
            "quality_score",
            "quality_grade",
            "quality_flags",
            "training_status",
        ]
    ].head(20)
)

,file_name,return_type,tax_year,quality_score,quality_grade,quality_flags,training_status
0,f1065--2025.pdf,1065,2025,100,good,none,eligible
1,f1065x--2025.pdf,1065,2025,100,good,none,eligible
2,p4163.pdf,unknown,<NA>,80,good,return_type_review;tax_year_review,review
3,p4164.pdf,unknown,<NA>,80,good,return_type_review;tax_year_review,review
4,i1065--2025.pdf,1065,2025,100,good,none,eligible
5,i1065x--2025.pdf,1065,2025,100,good,none,eligible
6,p541--2025.pdf,1065,2025,100,good,none,eligible
7,i1065s23--2025.pdf,1065,2025,100,good,none,eligible
8,i1065sb2--2018.pdf,1065,2018,100,good,none,eligible
9,i1065sd--2025.pdf,1065,2025,100,good,none,eligible


### Review and save

In [54]:
print("Quality-grade summary:")

display(
    document_inventory["quality_grade"]
    .value_counts(dropna=False)
    .rename_axis("quality_grade")
    .reset_index(name="document_count")
)

print("Training-status summary:")

display(
    document_inventory["training_status"]
    .value_counts(dropna=False)
    .rename_axis("training_status")
    .reset_index(name="document_count")
)

review_documents = document_inventory[
    document_inventory["training_status"] == "review"
]

excluded_documents = document_inventory[
    document_inventory["training_status"] == "exclude"
]

print(f"Eligible documents : {(document_inventory['training_status'] == 'eligible').sum()}")
print(f"Documents to review: {len(review_documents)}")
print(f"Documents excluded : {len(excluded_documents)}")

if not review_documents.empty:
    display(
        review_documents[
            [
                "file_name",
                "quality_score",
                "quality_flags",
            ]
        ].sort_values("quality_score").head(30)
    )

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

quality_table.to_csv(
    REPORTS_DIR / "document_quality_report.csv",
    index=False,
)

review_documents.to_csv(
    REPORTS_DIR / "documents_requiring_review.csv",
    index=False,
)

excluded_documents.to_csv(
    REPORTS_DIR / "excluded_documents.csv",
    index=False,
)

print("Document-quality reports saved successfully.")

Quality-grade summary:


,quality_grade,document_count
0,good,56
1,poor,2


Training-status summary:


,training_status,document_count
0,eligible,41
1,review,15
2,exclude,2


Eligible documents : 41
Documents to review: 15
Documents excluded : 2


,file_name,quality_score,quality_flags
2,p4163.pdf,80,return_type_review;tax_year_review
3,p4164.pdf,80,return_type_review;tax_year_review
56,f1120utp--2022.pdf,80,low_alphabetic_ratio
11,i1065sk3--2025.pdf,90,return_type_review
25,f4562--2025.pdf,90,return_type_review
28,f7004--2018.pdf,90,return_type_review
29,f851--2016.pdf,90,return_type_review
21,sources_1065_web.csv,90,return_type_review
32,i4562--2025.pdf,90,return_type_review
33,i4626--2025.pdf,90,return_type_review


Document-quality reports saved successfully.


## Section 14: Re-detect Form Type and Tax Year from Full Text

This section uses the complete cleaned text to improve form-type and tax-year detection. Original values are preserved for comparison, and uncertain results are flagged for review.

In [55]:
FULL_FORM_PATTERNS = {
    "1065": re.compile(
        r"\bform[\s_-]*1065(?!\d)",
        re.IGNORECASE,
    ),
    "1120": re.compile(
        r"\bform[\s_-]*1120(?![\s_-]?[a-z]|\d)",
        re.IGNORECASE,
    ),
}


def capped_match_count(pattern, text, maximum=20):
    return min(
        len(list(pattern.finditer(text))),
        maximum,
    )


def detect_form_type_from_full_text(file_path, clean_text):
    file_name_text = (
        file_path.stem
        .replace("_", " ")
        .replace("-", " ")
    )

    # The beginning usually contains the form title
    opening_text = clean_text[:10_000]

    scores = {}

    for form_type, pattern in FULL_FORM_PATTERNS.items():
        file_name_matches = capped_match_count(
            pattern,
            file_name_text,
            maximum=2,
        )

        opening_matches = capped_match_count(
            pattern,
            opening_text,
            maximum=5,
        )

        full_text_matches = capped_match_count(
            pattern,
            clean_text,
            maximum=20,
        )

        scores[form_type] = (
            file_name_matches * 10
            + opening_matches * 5
            + full_text_matches
        )

    ranked_forms = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    best_form, best_score = ranked_forms[0]
    second_score = ranked_forms[1][1]

    if best_score == 0:
        detected_form = "unknown"
        status = "not_detected"

    elif best_score - second_score < 3:
        detected_form = "both"
        status = "ambiguous"

    else:
        detected_form = best_form
        status = "detected"

    return {
        "full_text_return_type": detected_form,
        "full_text_return_type_status": status,
        "form_1065_score": scores["1065"],
        "form_1120_score": scores["1120"],
    }

### Detect the tax year from complete text

In [56]:
FULL_TEXT_YEAR_PATTERN = (
    r"(?:19[9][0-9]|20[0-9]{2})"
)


def add_year_score(
    year_scores,
    year_sources,
    year_value,
    points,
    source,
):
    year_number = int(year_value)

    if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
        year_scores[year_number] += points
        year_sources.setdefault(
            year_number,
            set(),
        ).add(source)


def detect_tax_year_from_full_text(file_path, clean_text):
    year_scores = Counter()
    year_sources = {}

    file_name_text = (
        file_path.stem
        .replace("_", " ")
        .replace("-", " ")
    )

    opening_text = clean_text[:15_000]

    # File-name years receive the highest weight
    for year in re.findall(
        rf"\b({FULL_TEXT_YEAR_PATTERN})\b",
        file_name_text,
    ):
        add_year_score(
            year_scores,
            year_sources,
            year,
            10,
            "file_name",
        )

    opening_patterns = [
        rf"\bfor\s+calendar\s+year\s+({FULL_TEXT_YEAR_PATTERN})\b",
        rf"\btax\s+year\s+({FULL_TEXT_YEAR_PATTERN})\b",
        rf"\b({FULL_TEXT_YEAR_PATTERN})\s+form\s+(?:1065|1120)\b",
        rf"\bform\s+(?:1065|1120)\s*\(?({FULL_TEXT_YEAR_PATTERN})\)?",
        rf"\b({FULL_TEXT_YEAR_PATTERN})\s+instructions?\s+for\s+form\b",
    ]

    for pattern in opening_patterns:
        for year in re.findall(
            pattern,
            opening_text,
            flags=re.IGNORECASE,
        ):
            add_year_score(
                year_scores,
                year_sources,
                year,
                6,
                "opening_text",
            )

    # Full-document matches receive less weight because tax
    # documents often refer to prior or future years.
    full_text_patterns = [
        rf"\bfor\s+tax\s+year\s+({FULL_TEXT_YEAR_PATTERN})\b",
        rf"\b({FULL_TEXT_YEAR_PATTERN})\s+form\s+(?:1065|1120)\b",
        rf"\bform\s+(?:1065|1120)\s*\(?({FULL_TEXT_YEAR_PATTERN})\)?",
    ]

    for pattern in full_text_patterns:
        matches = re.findall(
            pattern,
            clean_text,
            flags=re.IGNORECASE,
        )

        for year in matches[:20]:
            add_year_score(
                year_scores,
                year_sources,
                year,
                1,
                "full_text",
            )

    if not year_scores:
        return {
            "full_text_tax_year": None,
            "full_text_tax_year_source": "not_detected",
            "full_text_tax_year_status": "not_detected",
            "tax_year_candidates": "",
        }

    ranked_years = year_scores.most_common()
    best_year, best_score = ranked_years[0]

    tied_years = [
        year
        for year, score in ranked_years
        if score == best_score
    ]

    status = (
        "detected"
        if len(tied_years) == 1
        else "ambiguous"
    )

    candidate_summary = ";".join(
        f"{year}:{score}"
        for year, score in ranked_years[:5]
    )

    return {
        "full_text_tax_year": best_year,
        "full_text_tax_year_source": ", ".join(
            sorted(year_sources.get(best_year, {"unknown"}))
        ),
        "full_text_tax_year_status": status,
        "tax_year_candidates": candidate_summary,
    }

### Process every cleaned document

In [57]:
redetection_results = []

documents_to_redetect = document_inventory[
    document_inventory["clean_text_path"].notna()
].copy()

for row in tqdm(
    documents_to_redetect.itertuples(index=False),
    total=len(documents_to_redetect),
    desc="Re-detecting document metadata",
):
    try:
        file_path = file_lookup[row.relative_path]
        clean_text_path = OUTPUT_DIR / row.clean_text_path

        clean_text = clean_text_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        form_result = detect_form_type_from_full_text(
            file_path,
            clean_text,
        )

        year_result = detect_tax_year_from_full_text(
            file_path,
            clean_text,
        )

        redetection_results.append(
            {
                "document_id": row.document_id,
                **form_result,
                **year_result,
                "redetection_error": None,
            }
        )

    except Exception as error:
        redetection_results.append(
            {
                "document_id": row.document_id,
                "full_text_return_type": "unknown",
                "full_text_return_type_status": "failed",
                "form_1065_score": 0,
                "form_1120_score": 0,
                "full_text_tax_year": None,
                "full_text_tax_year_source": "failed",
                "full_text_tax_year_status": "failed",
                "tax_year_candidates": "",
                "redetection_error": str(error),
            }
        )

redetection_table = pd.DataFrame(redetection_results)

print(f"Documents re-detected: {len(redetection_table)}")

if not redetection_table.empty:
    display(redetection_table.head())

Re-detecting document metadata:   0%|          | 0/58 [00:00<?, ?it/s]

Documents re-detected: 58


,document_id,full_text_return_type,full_text_return_type_status,form_1065_score,form_1120_score,full_text_tax_year,full_text_tax_year_source,full_text_tax_year_status,tax_year_candidates,redetection_error
0,d304db736face09f,1065,detected,45,0,2025,"file_name, full_text, opening_text",detected,2025:58,None
1,afade3aec3a942e0,1065,detected,38,0,2025,file_name,detected,2025:10,None
2,d1956ddbb121d7f5,both,ambiguous,17,16,2023,full_text,detected,2023:1,None
3,caa77758d1eecb37,both,ambiguous,20,19,2021,full_text,detected,2021:4;2025:2;2024:1;2026:1,None
4,e54829d5a97de257,1065,detected,45,2,2025,"file_name, full_text, opening_text",detected,2025:55;2026:1,None


### Compare with the original detection

In [58]:
# Preserve the original results
document_inventory["initial_return_type"] = (
    document_inventory["return_type"]
)

document_inventory["initial_tax_year"] = (
    document_inventory["tax_year"]
)

redetection_columns = [
    "document_id",
    "full_text_return_type",
    "full_text_return_type_status",
    "form_1065_score",
    "form_1120_score",
    "full_text_tax_year",
    "full_text_tax_year_source",
    "full_text_tax_year_status",
    "tax_year_candidates",
    "redetection_error",
]

document_inventory = document_inventory.drop(
    columns=redetection_columns[1:],
    errors="ignore",
)

document_inventory = document_inventory.merge(
    redetection_table[redetection_columns],
    on="document_id",
    how="left",
)

document_inventory["return_type_changed"] = (
    document_inventory["full_text_return_type_status"].eq("detected")
    & document_inventory["full_text_return_type"].ne(
        document_inventory["initial_return_type"]
    )
)

document_inventory["tax_year_changed"] = (
    document_inventory["full_text_tax_year_status"].eq("detected")
    & document_inventory["full_text_tax_year"].notna()
    & document_inventory["full_text_tax_year"].ne(
        document_inventory["initial_tax_year"]
    )
)

### Apply confident results

In [59]:
confident_form_mask = (
    document_inventory["full_text_return_type_status"]
    == "detected"
)

document_inventory.loc[
    confident_form_mask,
    "return_type",
] = document_inventory.loc[
    confident_form_mask,
    "full_text_return_type",
]

confident_year_mask = (
    document_inventory["full_text_tax_year_status"]
    == "detected"
) & document_inventory["full_text_tax_year"].notna()

document_inventory.loc[
    confident_year_mask,
    "tax_year",
] = document_inventory.loc[
    confident_year_mask,
    "full_text_tax_year",
]

document_inventory.loc[
    confident_year_mask,
    "tax_year_source",
] = document_inventory.loc[
    confident_year_mask,
    "full_text_tax_year_source",
]

document_inventory.loc[
    confident_year_mask,
    "tax_year_status",
] = "detected"

document_inventory["tax_year"] = pd.array(
    document_inventory["tax_year"],
    dtype="Int64",
)

### Review and save

In [60]:
redetection_review = document_inventory[
    document_inventory["full_text_return_type_status"].isin(
        ["ambiguous", "not_detected", "failed"]
    )
    | document_inventory["full_text_tax_year_status"].isin(
        ["ambiguous", "not_detected", "failed"]
    )
    | document_inventory["return_type_changed"]
    | document_inventory["tax_year_changed"]
].copy()

print(
    "Form types changed:",
    document_inventory["return_type_changed"].sum(),
)

print(
    "Tax years changed:",
    document_inventory["tax_year_changed"].sum(),
)

print(
    "Documents requiring metadata review:",
    len(redetection_review),
)

if not redetection_review.empty:
    display(
        redetection_review[
            [
                "file_name",
                "initial_return_type",
                "return_type",
                "initial_tax_year",
                "tax_year",
                "full_text_return_type_status",
                "full_text_tax_year_status",
            ]
        ].head(30)
    )

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

redetection_table.to_csv(
    REPORTS_DIR / "full_text_metadata_detection.csv",
    index=False,
)

redetection_review.to_csv(
    REPORTS_DIR / "metadata_requiring_review.csv",
    index=False,
)

print("Full-text metadata results saved successfully.")

Form types changed: 7
Tax years changed: 0
Documents requiring metadata review: 17


,file_name,initial_return_type,return_type,initial_tax_year,tax_year,full_text_return_type_status,full_text_tax_year_status
2,p4163.pdf,unknown,unknown,<NA>,2023,ambiguous,detected
3,p4164.pdf,unknown,unknown,<NA>,2021,ambiguous,detected
11,i1065sk3--2025.pdf,both,1065,2025,2025,detected,detected
21,sources_1065_web.csv,unknown,unknown,2025,2025,not_detected,detected
25,f4562--2025.pdf,unknown,unknown,2025,2025,not_detected,detected
26,f4626--2025.pdf,1120,1120,2025,2025,ambiguous,detected
28,f7004--2018.pdf,both,both,2018,2018,ambiguous,detected
29,f851--2016.pdf,unknown,1120,2016,2016,detected,detected
32,i4562--2025.pdf,unknown,1065,2025,2025,detected,detected
33,i4626--2025.pdf,unknown,1120,2025,2025,detected,detected


Full-text metadata results saved successfully.


## Section 15: Split Documents into Retrieval Chunks

This section divides approved documents into smaller passages for semantic search. Each chunk retains its source document, form type, tax year, page number, and file path so the chatbot can provide citations.

In [61]:
CHUNKS_DIR = CLEANED_DIR / "retrieval_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE_WORDS = 180
CHUNK_OVERLAP_WORDS = 40
MIN_CHUNK_WORDS = 30

# Only automatically approved documents enter the knowledge base.
# Review documents can be added later after manual approval.
retrieval_documents = document_inventory[
    (document_inventory["training_status"] == "eligible")
    & (~document_inventory["exact_duplicate"])
    & (document_inventory["clean_text_path"].notna())
].copy()

print(f"Documents selected: {len(retrieval_documents)}")
print(f"Chunk size        : {CHUNK_SIZE_WORDS} words")
print(f"Chunk overlap     : {CHUNK_OVERLAP_WORDS} words")
print(f"Chunk output      : {CHUNKS_DIR}")

Documents selected: 41
Chunk size        : 180 words
Chunk overlap     : 40 words
Chunk output      : /kaggle/working/tax_llm/cleaned_documents/retrieval_chunks


### Separate text by page

In [62]:
PAGE_MARKER_PATTERN = re.compile(
    r"^--- Page (\d+) ---\s*$",
    re.MULTILINE,
)


def split_text_by_page(text):
    matches = list(PAGE_MARKER_PATTERN.finditer(text))

    # Non-PDF files may not contain page markers
    if not matches:
        return [
            {
                "page_number": None,
                "page_text": text.strip(),
            }
        ]

    pages = []

    for match_index, match in enumerate(matches):
        page_number = int(match.group(1))
        text_start = match.end()

        if match_index + 1 < len(matches):
            text_end = matches[match_index + 1].start()
        else:
            text_end = len(text)

        page_text = text[text_start:text_end].strip()

        pages.append(
            {
                "page_number": page_number,
                "page_text": page_text,
            }
        )

    return pages

### Define the chunking function

In [63]:
def split_page_into_chunks(
    page_text,
    chunk_size=CHUNK_SIZE_WORDS,
    overlap=CHUNK_OVERLAP_WORDS,
):
    words = page_text.split()

    if not words:
        return []

    chunks = []
    start_index = 0

    while start_index < len(words):
        end_index = min(
            start_index + chunk_size,
            len(words),
        )

        chunk_words = words[start_index:end_index]
        chunk_text = " ".join(chunk_words).strip()

        # Keep a short final chunk only when it is the entire page
        if (
            len(chunk_words) >= MIN_CHUNK_WORDS
            or start_index == 0
        ):
            chunks.append(
                {
                    "chunk_text": chunk_text,
                    "word_start": start_index,
                    "word_end": end_index,
                    "word_count": len(chunk_words),
                }
            )

        if end_index >= len(words):
            break

        start_index = end_index - overlap

    return chunks

### Create chunks and metadata

In [64]:
chunk_records = []
chunking_errors = []

for row in tqdm(
    retrieval_documents.itertuples(index=False),
    total=len(retrieval_documents),
    desc="Creating retrieval chunks",
):
    try:
        clean_text_file = OUTPUT_DIR / row.clean_text_path

        document_text = clean_text_file.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        pages = split_text_by_page(document_text)
        document_chunk_number = 0

        for page in pages:
            page_chunks = split_page_into_chunks(
                page["page_text"]
            )

            for page_chunk_number, chunk in enumerate(
                page_chunks,
                start=1,
            ):
                document_chunk_number += 1

                chunk_key = (
                    f"{row.document_id}|"
                    f"{page['page_number']}|"
                    f"{document_chunk_number}|"
                    f"{chunk['chunk_text']}"
                )

                chunk_id = hashlib.sha256(
                    chunk_key.encode("utf-8")
                ).hexdigest()[:24]

                chunk_records.append(
                    {
                        "chunk_id": chunk_id,
                        "document_id": row.document_id,
                        "document_chunk_number": (
                            document_chunk_number
                        ),
                        "page_chunk_number": page_chunk_number,
                        "page_number": page["page_number"],
                        "file_name": row.file_name,
                        "relative_path": row.relative_path,
                        "return_type": row.return_type,
                        "tax_year": row.tax_year,
                        "word_start": chunk["word_start"],
                        "word_end": chunk["word_end"],
                        "word_count": chunk["word_count"],
                        "chunk_text": chunk["chunk_text"],
                    }
                )

    except Exception as error:
        chunking_errors.append(
            {
                "document_id": row.document_id,
                "file_name": row.file_name,
                "chunking_error": str(error),
            }
        )

chunk_table = pd.DataFrame(chunk_records)

chunk_error_table = pd.DataFrame(
    chunking_errors,
    columns=[
        "document_id",
        "file_name",
        "chunking_error",
    ],
)

print(f"Chunks created : {len(chunk_table)}")
print(f"Chunking errors: {len(chunk_error_table)}")

if not chunk_table.empty:
    display(chunk_table.head())

Creating retrieval chunks:   0%|          | 0/41 [00:00<?, ?it/s]

Chunks created : 2411
Chunking errors: 0


,chunk_id,document_id,document_chunk_number,page_chunk_number,page_number,file_name,relative_path,return_type,tax_year,word_start,word_end,word_count,chunk_text
0,04894d5519cbe603f2a34d52,d304db736face09f,1,1,1.0,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,1065,2025,0,180,180,Form 1065 2025 U.S. Return of Partnership Inco...
1,8eb61cf4f9bdefbf40cfd6fb,d304db736face09f,2,2,1.0,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,1065,2025,140,320,180,year: J Check if Schedules C and M-3 are attac...
2,655358f7638c6f1146a8e8a3,d304db736face09f,3,3,1.0,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,1065,2025,280,460,180,. . . . . . . . 3 4 Ordinary income (loss) fro...
3,b7a87de951ef6af0aae3e8f2,d304db736face09f,4,4,1.0,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,1065,2025,420,600,180,wages (other than to partners) (less employmen...
4,3f49864d12ed6bbb28e820f4,d304db736face09f,5,5,1.0,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,1065,2025,560,740,180,. 13 14 Taxes and licenses . . . . . . . . . ....


### Validate the chunks

In [65]:
if not chunk_table.empty:
    chunk_summary = (
        chunk_table.groupby(
            ["return_type", "tax_year"],
            dropna=False,
        )
        .agg(
            document_count=("document_id", "nunique"),
            chunk_count=("chunk_id", "count"),
            average_words=("word_count", "mean"),
            minimum_words=("word_count", "min"),
            maximum_words=("word_count", "max"),
        )
        .reset_index()
    )

    chunk_summary["average_words"] = (
        chunk_summary["average_words"].round(1)
    )

    print("Chunk summary:")
    display(chunk_summary)

    duplicate_chunk_ids = (
        chunk_table["chunk_id"].duplicated().sum()
    )

    empty_chunks = (
        chunk_table["chunk_text"].str.strip().eq("").sum()
    )

    print(f"Duplicate chunk IDs: {duplicate_chunk_ids}")
    print(f"Empty chunks       : {empty_chunks}")

Chunk summary:


,return_type,tax_year,document_count,chunk_count,average_words,minimum_words,maximum_words
0,1065,2014,1,3,151.7,95,180
1,1065,2018,2,16,159.8,99,180
2,1065,2019,1,7,159.4,108,180
3,1065,2021,1,14,165.5,108,180
4,1065,2023,1,169,170.4,44,180
5,1065,2025,13,1629,168.7,41,180
6,1120,2011,2,23,167.8,79,180
7,1120,2015,1,10,168.6,119,180
8,1120,2016,2,29,163.3,42,180
9,1120,2018,4,66,167.6,47,180


Duplicate chunk IDs: 0
Empty chunks       : 0


### Preview a chunk with its citation

In [66]:
if not chunk_table.empty:
    sample_chunk = chunk_table.iloc[0]

    print("Sample citation metadata")
    print("------------------------")
    print(f"File     : {sample_chunk['file_name']}")
    print(f"Form     : {sample_chunk['return_type']}")
    print(f"Tax year : {sample_chunk['tax_year']}")
    print(f"Page     : {sample_chunk['page_number']}")
    print(f"Chunk ID : {sample_chunk['chunk_id']}")
    print()
    print(sample_chunk["chunk_text"][:2_000])

Sample citation metadata
------------------------
File     : f1065--2025.pdf
Form     : 1065
Tax year : 2025
Page     : 1.0
Chunk ID : 04894d5519cbe603f2a34d52

Form 1065 2025 U.S. Return of Partnership Income Department of the Treasury Internal Revenue Service Go to www.irs.gov/Form1065 for instructions and the latest information. OMB No. 1545-0123 For calendar year 2025, or tax year beginning , 2025, ending , 20 . Name of partnership Number and street Room or suite no. City or town State or province Country ZIP or foreign postal code A Principal business activity B Principal product or service C Business code number D Employer identification number E Date business started F Total assets (see instructions) $ G Check applicable boxes: (1) Initial return (2) Final return (3) Name change (4) Address change (5) Amended return H Check accounting method: (1) Cash (2) Accrual (3) Other (specify): I Number of Schedules K-1. Attach one for each person who was a partner at any time during the t

### Save the retrieval dataset

In [67]:
CHUNK_COLUMNS = [
    "chunk_id",
    "document_id",
    "document_chunk_number",
    "page_chunk_number",
    "page_number",
    "file_name",
    "relative_path",
    "return_type",
    "tax_year",
    "word_start",
    "word_end",
    "word_count",
    "chunk_text",
]

if chunk_table.empty:
    chunk_table = pd.DataFrame(columns=CHUNK_COLUMNS)
else:
    chunk_table = chunk_table[CHUNK_COLUMNS]

chunk_table.to_json(
    CHUNKS_DIR / "retrieval_chunks.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

chunk_table.drop(
    columns=["chunk_text"],
    errors="ignore",
).to_csv(
    REPORTS_DIR / "retrieval_chunk_inventory.csv",
    index=False,
)

chunk_error_table.to_csv(
    REPORTS_DIR / "chunking_errors.csv",
    index=False,
)

print("Retrieval chunks saved successfully.")
print(
    "JSONL file:",
    CHUNKS_DIR / "retrieval_chunks.jsonl",
)

Retrieval chunks saved successfully.
JSONL file: /kaggle/working/tax_llm/cleaned_documents/retrieval_chunks/retrieval_chunks.jsonl


## Section 16: Create Embeddings and Build the Vector Index

This section creates semantic embeddings for the approved retrieval chunks and stores them in a FAISS vector index. The index will allow the chatbot to find tax passages related to a user's question.

In [68]:
%pip install -q sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [69]:
import json

from datetime import datetime, timezone

import faiss
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer


LOCAL_EMBEDDING_MODEL = (
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
)

VECTOR_DIR = CLEANED_DIR / "vector_index"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_FILE = CHUNKS_DIR / "retrieval_chunks.jsonl"

FAISS_INDEX_PATH = VECTOR_DIR / "tax_retrieval.index"
EMBEDDING_MATRIX_PATH = VECTOR_DIR / "embedding_matrix.npy"
VECTOR_METADATA_PATH = VECTOR_DIR / "vector_metadata.jsonl"
VECTOR_METADATA_CSV_PATH = (
    REPORTS_DIR / "vector_metadata.csv"
)

if (
    "chunk_table" not in globals()
    or chunk_table.empty
):
    chunk_table = pd.read_json(
        CHUNK_FILE,
        orient="records",
        lines=True,
    )

chunk_table = chunk_table.reset_index(drop=True)

if chunk_table.empty:
    raise ValueError(
        "No chunks found. Run Section 15 first."
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Embedding model:", LOCAL_EMBEDDING_MODEL)
print("Device         :", device)
print("Chunks         :", f"{len(chunk_table):,}")

Embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Device         : cuda
Chunks         : 2,411


### Load the free model

In [70]:
embedding_model = SentenceTransformer(
    LOCAL_EMBEDDING_MODEL,
    device=device,
)

EMBEDDING_DIMENSIONS = (
    embedding_model.get_sentence_embedding_dimension()
)

print(
    "Embedding dimensions:",
    EMBEDDING_DIMENSIONS,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384


/tmp/ipykernel_139/2482645195.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


### Generate embeddings locally

In [71]:
chunk_texts = (
    chunk_table["chunk_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

if any(not text.strip() for text in chunk_texts):
    raise ValueError(
        "One or more chunks are empty. "
        "Review Section 15."
    )

embedding_matrix = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embedding_matrix = np.asarray(
    embedding_matrix,
    dtype=np.float32,
)

expected_shape = (
    len(chunk_table),
    EMBEDDING_DIMENSIONS,
)

if embedding_matrix.shape != expected_shape:
    raise ValueError(
        "Unexpected embedding shape: "
        f"{embedding_matrix.shape}"
    )

np.save(
    EMBEDDING_MATRIX_PATH,
    embedding_matrix,
)

print(
    "Embeddings generated:",
    f"{len(embedding_matrix):,}",
)

print(
    "Matrix shape:",
    embedding_matrix.shape,
)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embeddings generated: 2,411
Matrix shape: (2411, 384)


### Build the local FAISS index

In [72]:
vector_index = faiss.IndexFlatIP(
    EMBEDDING_DIMENSIONS
)

vector_index.add(embedding_matrix)

faiss.write_index(
    vector_index,
    str(FAISS_INDEX_PATH),
)

print(
    "Vectors indexed:",
    f"{vector_index.ntotal:,}",
)

print(
    "FAISS index:",
    FAISS_INDEX_PATH,
)

Vectors indexed: 2,411
FAISS index: /kaggle/working/tax_llm/cleaned_documents/vector_index/tax_retrieval.index


### save the meata data

In [73]:
vector_metadata = chunk_table.copy()

vector_metadata.insert(
    0,
    "vector_position",
    np.arange(
        len(vector_metadata),
        dtype=int,
    ),
)

vector_metadata["embedding_model"] = (
    LOCAL_EMBEDDING_MODEL
)

vector_metadata["embedding_dimensions"] = (
    EMBEDDING_DIMENSIONS
)

vector_metadata.to_json(
    VECTOR_METADATA_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

vector_metadata.drop(
    columns=["chunk_text"],
    errors="ignore",
).to_csv(
    VECTOR_METADATA_CSV_PATH,
    index=False,
)

index_configuration = {
    "embedding_model": LOCAL_EMBEDDING_MODEL,
    "embedding_dimensions": EMBEDDING_DIMENSIONS,
    "embedding_provider": "local_sentence_transformers",
    "similarity_metric": "cosine",
    "faiss_index_type": "IndexFlatIP",
    "vector_count": int(vector_index.ntotal),
    "api_cost": 0,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

(
    VECTOR_DIR / "index_configuration.json"
).write_text(
    json.dumps(
        index_configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print("Local vector index saved successfully.")

Local vector index saved successfully.


### Test the free semantic search

In [74]:
def search_vector_index(query, top_k=5):
    query_vector = embedding_model.encode(
        [str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_vector = np.asarray(
        query_vector,
        dtype=np.float32,
    )

    scores, positions = vector_index.search(
        query_vector,
        min(top_k, vector_index.ntotal),
    )

    results = []

    for score, position in zip(
        scores[0],
        positions[0],
    ):
        if position < 0:
            continue

        result = vector_metadata.iloc[
            int(position)
        ].to_dict()

        result["similarity_score"] = round(
            float(score),
            4,
        )

        results.append(result)

    return pd.DataFrame(results)

In [75]:
test_results = search_vector_index(
    "How is ordinary business income reported on Form 1065?",
    top_k=5,
)

display(
    test_results[
        [
            "similarity_score",
            "file_name",
            "return_type",
            "tax_year",
            "page_number",
            "chunk_text",
        ]
    ]
)

,similarity_score,file_name,return_type,tax_year,page_number,chunk_text
0,0.6708,i1065--2025.pdf,1065,2025,4.0,taxable income on an attached statement to For...
1,0.6531,i1065s23--2025.pdf,1065,2025,13.0,Form 1116 only requires reporting of total gro...
2,0.6301,i1065sk1--2025.pdf,1065,2025,14.0,it has attached a statement providing addition...
3,0.6211,i1065--2025.pdf,1065,2025,38.0,any amount reported on line 6c. On the line to...
4,0.6080,i1065--2025.pdf,1065,2025,14.0,"business includes (and portfolio income, there..."


## Section 17: Metadata-Aware Retrieval and Citation Formatting

This section improves semantic search by filtering passages by tax form and tax year, removing duplicate results, and formatting source citations.

In [76]:
from pathlib import Path
import json
import re
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


# Recreate the saved-file paths

VECTOR_DIR = Path(CLEANED_DIR) / "vector_index"

FAISS_INDEX_PATH = VECTOR_DIR / "tax_retrieval.index"
EMBEDDING_MATRIX_PATH = VECTOR_DIR / "embedding_matrix.npy"
VECTOR_METADATA_PATH = VECTOR_DIR / "vector_metadata.jsonl"
INDEX_CONFIGURATION_PATH = (
    VECTOR_DIR / "index_configuration.json"
)


# Confirm that Section 16 saved all required files

required_files = [
    FAISS_INDEX_PATH,
    EMBEDDING_MATRIX_PATH,
    VECTOR_METADATA_PATH,
    INDEX_CONFIGURATION_PATH,
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing Section 16 files:\n"
        + "\n".join(str(path) for path in missing_files)
        + "\nRerun the relevant Section 16 saving cells."
    )


# Load the index, embeddings, and metadata

vector_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

embedding_matrix = np.load(
    EMBEDDING_MATRIX_PATH
).astype(np.float32)

vector_metadata = pd.read_json(
    VECTOR_METADATA_PATH,
    orient="records",
    lines=True,
).reset_index(drop=True)


# Reload the same local embedding model used in Section 16
index_configuration = json.loads(
    INDEX_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

LOCAL_EMBEDDING_MODEL = index_configuration.get(
    "embedding_model",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = SentenceTransformer(
    LOCAL_EMBEDDING_MODEL,
    device=device,
)


# Validate that all saved components agree
if len(vector_metadata) != vector_index.ntotal:
    raise ValueError(
        "Vector metadata and FAISS index sizes do not match."
    )

if len(embedding_matrix) != len(vector_metadata):
    raise ValueError(
        "Embedding matrix and metadata sizes do not match."
    )

if embedding_matrix.shape[1] != vector_index.d:
    raise ValueError(
        "Embedding dimensions and FAISS dimensions do not match."
    )

print("Embedding model:", LOCAL_EMBEDDING_MODEL)
print("Device         :", device)
print("Vectors loaded :", f"{vector_index.ntotal:,}")
print("Metadata rows  :", f"{len(vector_metadata):,}")
print("Dimensions     :", vector_index.d)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Device         : cuda
Vectors loaded : 2,411
Metadata rows  : 2,411
Dimensions     : 384


In [77]:
print("Vector folder:", VECTOR_DIR)
print("FAISS index:", FAISS_INDEX_PATH)
print("Index exists:", FAISS_INDEX_PATH.exists())

Vector folder: /kaggle/working/tax_llm/cleaned_documents/vector_index
FAISS index: /kaggle/working/tax_llm/cleaned_documents/vector_index/tax_retrieval.index
Index exists: True


In [78]:
from pathlib import Path

# Search for the index created in Section 16
search_roots = [
    Path("/content"),
    Path("/kaggle/working"),
]

index_matches = []

for root in search_roots:
    if root.exists():
        index_matches.extend(
            root.rglob("tax_retrieval.index")
        )

if not index_matches:
    raise FileNotFoundError(
        "tax_retrieval.index was not found. "
        "If it was saved in Google Drive, mount Drive first. "
        "Otherwise, the runtime restart deleted it and "
        "Section 16 must be rerun."
    )

FAISS_INDEX_PATH = index_matches[0]
VECTOR_DIR = FAISS_INDEX_PATH.parent
EMBEDDING_MATRIX_PATH = (
    VECTOR_DIR / "embedding_matrix.npy"
)
VECTOR_METADATA_PATH = (
    VECTOR_DIR / "vector_metadata.jsonl"
)
INDEX_CONFIGURATION_PATH = (
    VECTOR_DIR / "index_configuration.json"
)

print("Vector folder:", VECTOR_DIR)
print("FAISS index:", FAISS_INDEX_PATH)
print("Index exists:", FAISS_INDEX_PATH.exists())
print(
    "Embedding matrix exists:",
    EMBEDDING_MATRIX_PATH.exists(),
)
print(
    "Metadata exists:",
    VECTOR_METADATA_PATH.exists(),
)

Vector folder: /kaggle/working/tax_llm/cleaned_documents/vector_index
FAISS index: /kaggle/working/tax_llm/cleaned_documents/vector_index/tax_retrieval.index
Index exists: True
Embedding matrix exists: True
Metadata exists: True


In [79]:
from pathlib import Path
import json
import re

import faiss
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer


VECTOR_DIR = Path(
    "/kaggle/working/tax_llm/cleaned_documents/vector_index"
)

FAISS_INDEX_PATH = VECTOR_DIR / "tax_retrieval.index"
EMBEDDING_MATRIX_PATH = VECTOR_DIR / "embedding_matrix.npy"
VECTOR_METADATA_PATH = VECTOR_DIR / "vector_metadata.jsonl"
INDEX_CONFIGURATION_PATH = VECTOR_DIR / "index_configuration.json"


# Load saved Section 16 artifacts
vector_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

embedding_matrix = np.load(
    EMBEDDING_MATRIX_PATH
).astype(np.float32)

vector_metadata = pd.read_json(
    VECTOR_METADATA_PATH,
    orient="records",
    lines=True,
).reset_index(drop=True)


# Determine the embedding model
if INDEX_CONFIGURATION_PATH.exists():
    index_configuration = json.loads(
        INDEX_CONFIGURATION_PATH.read_text(
            encoding="utf-8"
        )
    )

    LOCAL_EMBEDDING_MODEL = index_configuration.get(
        "embedding_model",
        "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    )
else:
    LOCAL_EMBEDDING_MODEL = (
        "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
    )


device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    LOCAL_EMBEDDING_MODEL,
    device=device,
)


# Validate saved artifacts
assert vector_index.ntotal == len(vector_metadata), (
    "FAISS index and metadata counts do not match."
)

assert len(embedding_matrix) == len(vector_metadata), (
    "Embedding matrix and metadata counts do not match."
)

assert embedding_matrix.shape[1] == vector_index.d, (
    "Embedding dimensions do not match the FAISS index."
)


print("Embedding model:", LOCAL_EMBEDDING_MODEL)
print("Device         :", device)
print("Vectors loaded :", f"{vector_index.ntotal:,}")
print("Metadata rows  :", f"{len(vector_metadata):,}")
print("Dimensions     :", vector_index.d)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Device         : cuda
Vectors loaded : 2,411
Metadata rows  : 2,411
Dimensions     : 384


In [80]:
def normalize_return_type(value):
    if pd.isna(value):
        return None

    normalized = re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )

    if normalized.startswith("FORM"):
        normalized = normalized[4:]

    return normalized or None


def normalize_tax_year(value):
    if pd.isna(value):
        return None

    match = re.search(
        r"\b(?:19|20)\d{2}\b",
        str(value),
    )

    if match:
        return int(match.group())

    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


vector_metadata["_normalized_return_type"] = (
    vector_metadata["return_type"].map(
        normalize_return_type
    )
)

vector_metadata["_normalized_tax_year"] = (
    vector_metadata["tax_year"].map(
        normalize_tax_year
    )
)

print("Available forms:")
print(
    vector_metadata["_normalized_return_type"]
    .dropna()
    .value_counts()
    .head(20)
)

print("\nAvailable tax years:")
print(
    vector_metadata["_normalized_tax_year"]
    .dropna()
    .value_counts()
    .sort_index()
)

Available forms:
_normalized_return_type
1065    1838
1120     573
Name: count, dtype: int64

Available tax years:
_normalized_tax_year
2011      23
2014       3
2015      10
2016      29
2018      82
2019      22
2021      14
2022      57
2023     169
2024     172
2025    1830
Name: count, dtype: int64


### Detect overlapping chunks

In [81]:
def normalized_word_set(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            str(text).lower(),
        )
    )


def chunks_are_too_similar(
    first_text,
    second_text,
    threshold=0.85,
):
    first_words = normalized_word_set(first_text)
    second_words = normalized_word_set(second_text)

    if not first_words or not second_words:
        return False

    overlap = len(first_words & second_words)
    union = len(first_words | second_words)

    return (overlap / union) >= threshold


print("Overlapping-chunk detection is ready.")

Overlapping-chunk detection is ready.


### Build metadata-aware retrieval

In [82]:
def retrieve_chunks(
    query,
    top_k=5,
    return_type=None,
    tax_year=None,
    minimum_score=None,
    max_chunks_per_document=2,
):
    query = str(query).strip()

    if not query:
        raise ValueError("The search query cannot be empty.")

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    metadata_mask = pd.Series(
        True,
        index=vector_metadata.index,
    )

    normalized_form = normalize_return_type(return_type)
    normalized_year = normalize_tax_year(tax_year)

    if return_type is not None:
        metadata_mask &= (
            vector_metadata["_normalized_return_type"]
            == normalized_form
        )

    if tax_year is not None:
        metadata_mask &= (
            vector_metadata["_normalized_tax_year"]
            == normalized_year
        )

    candidate_positions = np.flatnonzero(
        metadata_mask.to_numpy()
    )

    if len(candidate_positions) == 0:
        return pd.DataFrame()

    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_vector = np.asarray(
        query_vector,
        dtype=np.float32,
    )[0]

    candidate_vectors = np.asarray(
        embedding_matrix[candidate_positions],
        dtype=np.float32,
    )

    similarity_scores = candidate_vectors @ query_vector

    ranked_indices = np.argsort(-similarity_scores)

    results = []
    document_counts = {}

    for ranked_index in ranked_indices:
        position = int(candidate_positions[ranked_index])
        score = float(similarity_scores[ranked_index])

        if minimum_score is not None and score < minimum_score:
            continue

        record = vector_metadata.iloc[position].to_dict()

        document_id = str(
            record.get("document_id")
            or record.get("file_name")
            or position
        )

        if (
            document_counts.get(document_id, 0)
            >= max_chunks_per_document
        ):
            continue

        candidate_text = str(
            record.get("chunk_text", "")
        )

        duplicate_found = any(
            chunks_are_too_similar(
                candidate_text,
                existing.get("chunk_text", ""),
            )
            for existing in results
        )

        if duplicate_found:
            continue

        record["similarity_score"] = round(score, 4)
        results.append(record)

        document_counts[document_id] = (
            document_counts.get(document_id, 0) + 1
        )

        if len(results) >= top_k:
            break

    result_table = pd.DataFrame(results)

    if not result_table.empty:
        result_table.insert(
            0,
            "result_number",
            range(1, len(result_table) + 1),
        )

    return result_table


print("Metadata-aware retrieval is ready.")

Metadata-aware retrieval is ready.


### Create readable citations

In [83]:
def clean_display_value(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    return text or None


def format_citation(record):
    citation_number = record.get(
        "result_number",
        "?",
    )

    parts = []

    file_name = clean_display_value(
        record.get("file_name")
    )
    return_type = clean_display_value(
        record.get("return_type")
    )
    tax_year = clean_display_value(
        record.get("tax_year")
    )
    page_number = clean_display_value(
        record.get("page_number")
    )

    if file_name:
        parts.append(file_name)

    if return_type:
        parts.append(f"Form {return_type}")

    if tax_year:
        parts.append(f"tax year {tax_year}")

    if page_number:
        parts.append(f"page {page_number}")

    return (
        f"[{citation_number}] "
        + " — ".join(parts)
    )


def display_retrieval_results(results):
    if results.empty:
        print(
            "No passages matched the selected filters."
        )
        return

    for _, record in results.iterrows():
        print(format_citation(record))
        print(
            "Similarity:",
            f"{record['similarity_score']:.4f}",
        )
        print()
        print(record["chunk_text"])
        print("\n" + "=" * 80 + "\n")


print("Citation formatting is ready.")

Citation formatting is ready.


In [84]:
def build_retrieval_context(results):
    if results.empty:
        return ""

    context_sections = []

    for _, record in results.iterrows():
        citation = format_citation(record)

        context_sections.append(
            f"{citation}\n"
            f"{record['chunk_text']}"
        )

    return "\n\n".join(context_sections)


print("Retrieval-context builder is ready.")

Retrieval-context builder is ready.


In [85]:
test_results = retrieve_chunks(
    query=(
        "How is ordinary business income "
        "reported on Form 1065?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
)

if test_results.empty:
    print(
        "No matching passages found. "
        "Check the available form and tax-year values."
    )
else:
    display(
        test_results[
            [
                "result_number",
                "similarity_score",
                "file_name",
                "return_type",
                "tax_year",
                "page_number",
            ]
        ]
    )

    display_retrieval_results(test_results)

,result_number,similarity_score,file_name,return_type,tax_year,page_number
0,1,0.6708,i1065--2025.pdf,1065,2025,4.0
1,2,0.6531,i1065s23--2025.pdf,1065,2025,13.0
2,3,0.6301,i1065sk1--2025.pdf,1065,2025,14.0
3,4,0.6211,i1065--2025.pdf,1065,2025,38.0
4,5,0.5906,i1065s23--2025.pdf,1065,2025,39.0


[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 4
Similarity: 0.6708

taxable income on an attached statement to Form 1065 in the same manner as a corporation. The organization may use Form 1120, U.S. Corporation Income Tax Return, for this purpose. Enter the organization’s taxable income, if any, on Form 1065, Schedule K, line 6a, and each member’s distributive share in box 6a of Schedule K-1 (Form 1065). Net operating losses aren’t deductible by the members but may be carried back or forward by the organization under the rules of section 172. The religious or apostolic organization must also make its annual information return available for public inspection. For this purpose, an annual information return includes an exact copy of Form 1065 and all accompanying schedules and attached statements, except Schedules K-1. For more details, see Regulations section 301.6104(d)-1. A qualifying syndicate, pool, joint venture, or similar organization may elect under section 761(a) not to

### Validate filters and ranking

In [86]:
if test_results.empty:
    print(
        "Validation could not run because no matching "
        "passages were retrieved."
    )

else:
    returned_forms = {
        normalize_return_type(value)
        for value in test_results["return_type"]
    }

    returned_years = {
        normalize_tax_year(value)
        for value in test_results["tax_year"]
    }

    assert returned_forms == {"1065"}, (
        f"Unexpected forms returned: {returned_forms}"
    )

    assert returned_years == {2025}, (
        f"Unexpected tax years returned: {returned_years}"
    )

    assert test_results[
        "similarity_score"
    ].is_monotonic_decreasing, (
        "Results are not ordered by decreasing similarity."
    )

    assert len(test_results) <= 5, (
        "More results were returned than requested."
    )

    print("Metadata filter validation passed.")

Metadata filter validation passed.


### Preview the chatbot evidence package

In [87]:
retrieval_context = build_retrieval_context(
    test_results
)

if not retrieval_context:
    print("No evidence package was created.")
else:
    print(retrieval_context[:5_000])
    print()
    print(
        "Evidence package length:",
        f"{len(retrieval_context):,} characters",
    )

[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 4
taxable income on an attached statement to Form 1065 in the same manner as a corporation. The organization may use Form 1120, U.S. Corporation Income Tax Return, for this purpose. Enter the organization’s taxable income, if any, on Form 1065, Schedule K, line 6a, and each member’s distributive share in box 6a of Schedule K-1 (Form 1065). Net operating losses aren’t deductible by the members but may be carried back or forward by the organization under the rules of section 172. The religious or apostolic organization must also make its annual information return available for public inspection. For this purpose, an annual information return includes an exact copy of Form 1065 and all accompanying schedules and attached statements, except Schedules K-1. For more details, see Regulations section 301.6104(d)-1. A qualifying syndicate, pool, joint venture, or similar organization may elect under section 761(a) not to be treated as a par

## Section 18: Cost-Free, Evidence-Grounded Answer Generator

This section uses a local FLAN-T5 model to generate answers only from retrieved tax-document evidence. No paid API is required.

In [88]:
from pathlib import Path

import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)


GENERATION_MODEL_NAME = "google/flan-t5-base"

MODEL_CACHE_DIR = Path(
    "/kaggle/working/tax_llm/model_cache"
)
MODEL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

generation_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

generation_tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL_NAME,
    cache_dir=str(MODEL_CACHE_DIR),
)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME,
    cache_dir=str(MODEL_CACHE_DIR),
)

generation_model = generation_model.to(
    generation_device
)
generation_model.eval()

print("Answer model :", GENERATION_MODEL_NAME)
print("Device       :", generation_device)
print("Model loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Answer model : google/flan-t5-base
Device       : cuda
Model loaded successfully.


### Build the evidence-constrained prompt:

In [89]:
def build_answer_prompt(question, retrieval_context):
    question = str(question).strip()
    retrieval_context = str(retrieval_context).strip()

    if not question:
        raise ValueError("The question cannot be empty.")

    if not retrieval_context:
        raise ValueError(
            "No retrieved evidence is available."
        )

    return f"""
Answer the tax question using only the evidence below.

Rules:
- Do not use outside knowledge.
- If the evidence is insufficient, say:
  "The retrieved documents do not provide enough information."
- Keep the answer concise and factual.
- Include supporting citation numbers such as [1] or [2].
- Do not invent citations.

Question:
{question}

Evidence:
{retrieval_context}

Answer:
""".strip()


test_question = (
    "How is ordinary business income "
    "reported on Form 1065?"
)

test_prompt = build_answer_prompt(
    test_question,
    retrieval_context,
)

print(test_prompt[:5_000])
print()
print(
    "Prompt length:",
    f"{len(test_prompt):,} characters",
)

Answer the tax question using only the evidence below.

Rules:
- Do not use outside knowledge.
- If the evidence is insufficient, say:
  "The retrieved documents do not provide enough information."
- Keep the answer concise and factual.
- Include supporting citation numbers such as [1] or [2].
- Do not invent citations.

Question:
How is ordinary business income reported on Form 1065?

Evidence:
[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 4
taxable income on an attached statement to Form 1065 in the same manner as a corporation. The organization may use Form 1120, U.S. Corporation Income Tax Return, for this purpose. Enter the organization’s taxable income, if any, on Form 1065, Schedule K, line 6a, and each member’s distributive share in box 6a of Schedule K-1 (Form 1065). Net operating losses aren’t deductible by the members but may be carried back or forward by the organization under the rules of section 172. The religious or apostolic organization must also make its annu

### Generate the grounded answer

In [90]:
def generate_grounded_answer(
    prompt,
    max_input_tokens=512,
    max_new_tokens=180,
):
    prompt = str(prompt).strip()

    if not prompt:
        raise ValueError("The prompt cannot be empty.")

    model_inputs = generation_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    ).to(generation_device)

    with torch.inference_mode():
        generated_tokens = generation_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.1,
            early_stopping=True,
        )

    answer = generation_tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True,
    ).strip()

    return answer


test_answer = generate_grounded_answer(
    test_prompt
)

print("Generated answer:")
print(test_answer)

Generated answer:
Form 1065, Schedule K.


### Create the complete tax-question pipeline

In [91]:
def answer_tax_question(
    question,
    return_type=None,
    tax_year=None,
    top_k=5,
    minimum_score=None,
):
    question = str(question).strip()

    if not question:
        raise ValueError(
            "The question cannot be empty."
        )

    # 1. Retrieve relevant document passages
    results = retrieve_chunks(
        query=question,
        return_type=return_type,
        tax_year=tax_year,
        top_k=top_k,
        minimum_score=minimum_score,
    )

    if results.empty:
        return {
            "question": question,
            "answer": (
                "No passages matched the selected "
                "form and tax-year filters."
            ),
            "sources": results,
            "prompt": None,
        }

    # 2. Package the passages as cited evidence
    context = build_retrieval_context(
        results
    )

    # 3. Build the evidence-constrained prompt
    prompt = build_answer_prompt(
        question,
        context,
    )

    # 4. Generate the answer locally
    answer = generate_grounded_answer(
        prompt
    )

    return {
        "question": question,
        "answer": answer,
        "sources": results,
        "prompt": prompt,
    }


print("Complete tax-question pipeline is ready.")

Complete tax-question pipeline is ready.


### Then test the complete pipeline

In [92]:
response = answer_tax_question(
    question=(
        "How is ordinary business income "
        "reported on Form 1065?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
)

print("Question:")
print(response["question"])

print("\nAnswer:")
print(response["answer"])

print("\nRetrieved sources:")
display_retrieval_results(
    response["sources"]
)

Question:
How is ordinary business income reported on Form 1065?

Answer:
Form 1065, Schedule K.

Retrieved sources:
[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 4
Similarity: 0.6708

taxable income on an attached statement to Form 1065 in the same manner as a corporation. The organization may use Form 1120, U.S. Corporation Income Tax Return, for this purpose. Enter the organization’s taxable income, if any, on Form 1065, Schedule K, line 6a, and each member’s distributive share in box 6a of Schedule K-1 (Form 1065). Net operating losses aren’t deductible by the members but may be carried back or forward by the organization under the rules of section 172. The religious or apostolic organization must also make its annual information return available for public inspection. For this purpose, an annual information return includes an exact copy of Form 1065 and all accompanying schedules and attached statements, except Schedules K-1. For more details, see Regulations section 301.

### Validate generated citations

In [93]:
def validate_answer_citations(answer, sources):
    answer = str(answer).strip()

    cited_numbers = {
        int(number)
        for number in re.findall(r"\[(\d+)\]", answer)
    }

    valid_numbers = set()

    if not sources.empty:
        valid_numbers = {
            int(number)
            for number in sources["result_number"]
        }

    invalid_numbers = cited_numbers - valid_numbers

    insufficient_answer = (
        "do not provide enough information"
        in answer.lower()
    )

    citations_valid = (
        not invalid_numbers
        and (
            bool(cited_numbers)
            or insufficient_answer
        )
    )

    return {
        "citations_valid": citations_valid,
        "cited_numbers": sorted(cited_numbers),
        "valid_numbers": sorted(valid_numbers),
        "invalid_numbers": sorted(invalid_numbers),
    }


citation_check = validate_answer_citations(
    response["answer"],
    response["sources"],
)

print("Citations valid :", citation_check["citations_valid"])
print("Citations used  :", citation_check["cited_numbers"])
print("Available       :", citation_check["valid_numbers"])
print("Invalid         :", citation_check["invalid_numbers"])

if not citation_check["citations_valid"]:
    print(
        "\nWarning: The generated answer contains missing "
        "or unsupported citation numbers."
    )

Citations valid : False
Citations used  : []
Available       : [1, 2, 3, 4, 5]
Invalid         : []



### Add retry and safe fallback handling

In [94]:
def build_citation_retry_prompt(
    question,
    retrieval_context,
    valid_numbers,
):
    allowed_citations = ", ".join(
        f"[{number}]"
        for number in valid_numbers
    )

    return f"""
Answer the tax question using only the supplied evidence.

Requirements:
- Cite every factual statement.
- Use only these citation numbers: {allowed_citations}
- Do not create other citation numbers.
- Do not use outside knowledge.
- If the evidence is insufficient, respond exactly:
  "The retrieved documents do not provide enough information."

Question:
{question}

Evidence:
{retrieval_context}

Answer:
""".strip()


def answer_tax_question(
    question,
    return_type=None,
    tax_year=None,
    top_k=5,
    minimum_score=None,
):
    question = str(question).strip()

    if not question:
        raise ValueError(
            "The question cannot be empty."
        )

    # Retrieve evidence
    results = retrieve_chunks(
        query=question,
        return_type=return_type,
        tax_year=tax_year,
        top_k=top_k,
        minimum_score=minimum_score,
    )

    if results.empty:
        return {
            "question": question,
            "answer": (
                "No passages matched the selected "
                "form and tax-year filters."
            ),
            "sources": results,
            "prompt": None,
            "citation_check": None,
            "generation_attempts": 0,
            "used_fallback": True,
        }

    context = build_retrieval_context(
        results
    )

    # First generation attempt
    prompt = build_answer_prompt(
        question,
        context,
    )

    answer = generate_grounded_answer(
        prompt
    )

    citation_check = validate_answer_citations(
        answer,
        results,
    )

    generation_attempts = 1

    # Retry once with stricter citation instructions
    if not citation_check["citations_valid"]:
        valid_numbers = [
            int(number)
            for number in results["result_number"]
        ]

        prompt = build_citation_retry_prompt(
            question,
            context,
            valid_numbers,
        )

        answer = generate_grounded_answer(
            prompt
        )

        citation_check = validate_answer_citations(
            answer,
            results,
        )

        generation_attempts = 2

    # Safe fallback if the retry still fails
    used_fallback = not citation_check["citations_valid"]

    if used_fallback:
        answer = (
            "The retrieved documents do not provide "
            "enough information."
        )

        citation_check = validate_answer_citations(
            answer,
            results,
        )

    return {
        "question": question,
        "answer": answer,
        "sources": results,
        "prompt": prompt,
        "citation_check": citation_check,
        "generation_attempts": generation_attempts,
        "used_fallback": used_fallback,
    }


print("Citation-safe answer pipeline is ready.")

Citation-safe answer pipeline is ready.


In [95]:
response = answer_tax_question(
    question=(
        "How is ordinary business income "
        "reported on Form 1065?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
)

print("Answer:")
print(response["answer"])

print(
    "\nGeneration attempts:",
    response["generation_attempts"],
)

print(
    "Used safe fallback:",
    response["used_fallback"],
)

print(
    "Citations valid:",
    response["citation_check"]["citations_valid"],
)

print(
    "Citations used:",
    response["citation_check"]["cited_numbers"],
)

Answer:
The retrieved documents do not provide enough information.

Generation attempts: 2
Used safe fallback: True
Citations valid: True
Citations used: []


### Test the three main outcomes

In [96]:
def show_test_result(test_name, response):
    print(f"\n{'=' * 70}")
    print(test_name)
    print(f"{'=' * 70}")
    print("Answer:", response["answer"])
    print("Sources:", len(response["sources"]))
    print("Attempts:", response["generation_attempts"])
    print("Fallback:", response["used_fallback"])

    if response["citation_check"] is not None:
        print(
            "Citations valid:",
            response["citation_check"]["citations_valid"],
        )


# Test 1: Sufficient evidence
sufficient_response = answer_tax_question(
    question=(
        "How is ordinary business income "
        "reported on Form 1065?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
)

show_test_result(
    "TEST 1 — Sufficient evidence",
    sufficient_response,
)

assert not sufficient_response["sources"].empty
assert sufficient_response["citation_check"]["citations_valid"]


# Test 2: Insufficient or irrelevant evidence
insufficient_response = answer_tax_question(
    question=(
        "What tax deduction applies to a business "
        "operating on the planet Mars?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
    minimum_score=0.95,
)

show_test_result(
    "TEST 2 — Insufficient evidence",
    insufficient_response,
)

assert (
    insufficient_response["sources"].empty
    or insufficient_response["used_fallback"]
)


# Test 3: Invalid metadata filters
invalid_filter_response = answer_tax_question(
    question="How is business income reported?",
    return_type="9999",
    tax_year=1900,
    top_k=5,
)

show_test_result(
    "TEST 3 — Invalid form/year filters",
    invalid_filter_response,
)

assert invalid_filter_response["sources"].empty
assert invalid_filter_response["used_fallback"]

print("\nAll Section 18 pipeline tests passed.")


TEST 1 — Sufficient evidence
Answer: The retrieved documents do not provide enough information.
Sources: 5
Attempts: 2
Fallback: True
Citations valid: True

TEST 2 — Insufficient evidence
Answer: No passages matched the selected form and tax-year filters.
Sources: 0
Attempts: 0
Fallback: True

TEST 3 — Invalid form/year filters
Answer: No passages matched the selected form and tax-year filters.
Sources: 0
Attempts: 0
Fallback: True

All Section 18 pipeline tests passed.


## Section 19: Interactive Tax Chatbot

This section provides a simple interface for asking evidence-grounded tax questions and reviewing the retrieved sources.

In [97]:
def format_sources_for_interface(sources):
    if sources is None or sources.empty:
        return "No supporting passages were retrieved."

    sections = []

    for _, record in sources.iterrows():
        citation = format_citation(record)
        score = record.get("similarity_score", 0)
        passage = str(record.get("chunk_text", "")).strip()

        sections.append(
            f"### {citation}\n"
            f"**Similarity:** {score:.4f}\n\n"
            f"{passage}"
        )

    return "\n\n---\n\n".join(sections)


def run_tax_chatbot(
    question,
    return_type,
    tax_year,
    top_k,
):
    question = str(question).strip()

    if not question:
        return (
            "Please enter a tax question.",
            "No sources to display.",
        )

    selected_form = (
        None
        if return_type in (None, "", "All")
        else return_type
    )

    selected_year = (
        None
        if tax_year in (None, "", "All")
        else int(tax_year)
    )

    try:
        response = answer_tax_question(
            question=question,
            return_type=selected_form,
            tax_year=selected_year,
            top_k=int(top_k),
        )

        answer = response["answer"]

        status_lines = [
            answer,
            "",
            "---",
            f"Generation attempts: "
            f"{response['generation_attempts']}",
            f"Safe fallback used: "
            f"{response['used_fallback']}",
        ]

        if response["citation_check"] is not None:
            status_lines.append(
                "Citations valid: "
                f"{response['citation_check']['citations_valid']}"
            )

        answer_output = "\n".join(status_lines)

        sources_output = format_sources_for_interface(
            response["sources"]
        )

        return answer_output, sources_output

    except Exception as error:
        return (
            f"An error occurred: {type(error).__name__}: {error}",
            "No sources to display.",
        )


print("Chatbot interface handler is ready.")

Chatbot interface handler is ready.


### Create and launch the Gradio interface

In [98]:
import gradio as gr


# Build dropdown choices from the indexed metadata
available_forms = sorted(
    vector_metadata["_normalized_return_type"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

available_years = sorted(
    vector_metadata["_normalized_tax_year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist(),
    reverse=True,
)


with gr.Blocks(
    title="Evidence-Grounded Tax Chatbot"
) as tax_chatbot:
    gr.Markdown(
        """
        # Evidence-Grounded Tax Chatbot

        Ask questions using the indexed tax documents.
        Answers are generated locally and include supporting sources.
        """
    )

    with gr.Row():
        return_type_input = gr.Dropdown(
            choices=["All"] + available_forms,
            value="All",
            label="Tax form",
        )

        tax_year_input = gr.Dropdown(
            choices=["All"] + available_years,
            value="All",
            label="Tax year",
        )

        top_k_input = gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Number of passages",
        )

    question_input = gr.Textbox(
        label="Tax question",
        placeholder=(
            "Example: How is ordinary business income "
            "reported on Form 1065?"
        ),
        lines=3,
    )

    submit_button = gr.Button(
        "Ask the chatbot",
        variant="primary",
    )

    answer_output = gr.Markdown(
        label="Answer"
    )

    sources_output = gr.Markdown(
        label="Supporting sources"
    )

    submit_button.click(
        fn=run_tax_chatbot,
        inputs=[
            question_input,
            return_type_input,
            tax_year_input,
            top_k_input,
        ],
        outputs=[
            answer_output,
            sources_output,
        ],
    )

    question_input.submit(
        fn=run_tax_chatbot,
        inputs=[
            question_input,
            return_type_input,
            tax_year_input,
            top_k_input,
        ],
        outputs=[
            answer_output,
            sources_output,
        ],
    )


tax_chatbot.queue().launch(
    inline=True,
    share=False,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Validate the interface and pipeline

In [99]:
# Test 1: Empty question
empty_answer, empty_sources = run_tax_chatbot(
    question="",
    return_type="All",
    tax_year="All",
    top_k=5,
)

assert "Please enter" in empty_answer
assert empty_sources == "No sources to display."


# Test 2: Invalid filters
invalid_answer, invalid_sources = run_tax_chatbot(
    question="How is business income reported?",
    return_type="9999",
    tax_year=1900,
    top_k=5,
)

assert isinstance(invalid_answer, str)
assert isinstance(invalid_sources, str)
assert "An error occurred" not in invalid_answer


# Test 3: Valid Form 1065 question
valid_answer, valid_sources = run_tax_chatbot(
    question=(
        "How is ordinary business income "
        "reported on Form 1065?"
    ),
    return_type="1065",
    tax_year=2025,
    top_k=5,
)

assert isinstance(valid_answer, str)
assert valid_answer.strip()
assert "An error occurred" not in valid_answer
assert isinstance(valid_sources, str)
assert valid_sources.strip()
assert "Similarity:" in valid_sources


print("All Section 19 interface tests passed.")
print("\nSample answer:\n")
print(valid_answer)

All Section 19 interface tests passed.

Sample answer:

The retrieved documents do not provide enough information.

---
Generation attempts: 2
Safe fallback used: True
Citations valid: True


## Section 20: End-to-End Evaluation

This section evaluates retrieval success, metadata filtering, citation validity,
answer generation, and safe failure handling.

In [100]:
evaluation_cases = [
    {
        "case_id": "valid_1065_income",
        "question": (
            "How is ordinary business income "
            "reported on Form 1065?"
        ),
        "return_type": "1065",
        "tax_year": 2025,
        "top_k": 5,
        "minimum_score": None,
        "expect_sources": True,
        "expect_valid_citations": True,
    },
    {
        "case_id": "valid_1065_filing",
        "question": (
            "Who must file Form 1065?"
        ),
        "return_type": "1065",
        "tax_year": 2025,
        "top_k": 5,
        "minimum_score": None,
        "expect_sources": True,
        "expect_valid_citations": True,
    },
    {
        "case_id": "irrelevant_question",
        "question": (
            "What deduction applies to a business "
            "operating on Mars?"
        ),
        "return_type": "1065",
        "tax_year": 2025,
        "top_k": 5,
        "minimum_score": 0.95,
        "expect_sources": False,
        "expect_valid_citations": False,
    },
    {
        "case_id": "invalid_filters",
        "question": "How is business income reported?",
        "return_type": "9999",
        "tax_year": 1900,
        "top_k": 5,
        "minimum_score": None,
        "expect_sources": False,
        "expect_valid_citations": False,
    },
]

print(
    "Evaluation cases prepared:",
    len(evaluation_cases),
)

for case in evaluation_cases:
    print(
        case["case_id"],
        "—",
        case["question"],
    )

Evaluation cases prepared: 4
valid_1065_income — How is ordinary business income reported on Form 1065?
valid_1065_filing — Who must file Form 1065?
irrelevant_question — What deduction applies to a business operating on Mars?
invalid_filters — How is business income reported?


### Execute cases and collect metrics:

In [101]:
import time

import pandas as pd


evaluation_rows = []
evaluation_responses = {}


for case in evaluation_cases:
    print(f"Running: {case['case_id']}")

    start_time = time.perf_counter()

    try:
        response = answer_tax_question(
            question=case["question"],
            return_type=case["return_type"],
            tax_year=case["tax_year"],
            top_k=case["top_k"],
            minimum_score=case["minimum_score"],
        )

        elapsed_seconds = time.perf_counter() - start_time

        sources = response["sources"]
        source_count = len(sources)

        citation_check = response.get("citation_check")

        citations_valid = (
            citation_check["citations_valid"]
            if citation_check is not None
            else False
        )

        cited_numbers = (
            citation_check["cited_numbers"]
            if citation_check is not None
            else []
        )

        sources_expectation_passed = (
            (source_count > 0)
            == case["expect_sources"]
        )

        citations_expectation_passed = (
            citations_valid
            == case["expect_valid_citations"]
        )

        case_passed = (
            sources_expectation_passed
            and citations_expectation_passed
        )

        evaluation_responses[
            case["case_id"]
        ] = response

        evaluation_rows.append(
            {
                "case_id": case["case_id"],
                "source_count": source_count,
                "citations_valid": citations_valid,
                "cited_numbers": cited_numbers,
                "generation_attempts": response[
                    "generation_attempts"
                ],
                "used_fallback": response[
                    "used_fallback"
                ],
                "elapsed_seconds": round(
                    elapsed_seconds,
                    2,
                ),
                "sources_check": (
                    sources_expectation_passed
                ),
                "citations_check": (
                    citations_expectation_passed
                ),
                "case_passed": case_passed,
                "error": None,
            }
        )

    except Exception as error:
        elapsed_seconds = time.perf_counter() - start_time

        evaluation_rows.append(
            {
                "case_id": case["case_id"],
                "source_count": 0,
                "citations_valid": False,
                "cited_numbers": [],
                "generation_attempts": 0,
                "used_fallback": True,
                "elapsed_seconds": round(
                    elapsed_seconds,
                    2,
                ),
                "sources_check": False,
                "citations_check": False,
                "case_passed": False,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )


evaluation_results = pd.DataFrame(
    evaluation_rows
)

display(evaluation_results)

passed_cases = int(
    evaluation_results["case_passed"].sum()
)

total_cases = len(evaluation_results)

print(
    f"\nCases passed: {passed_cases}/{total_cases}"
)

print(
    "Overall pass rate:",
    f"{passed_cases / total_cases:.1%}",
)

Running: valid_1065_income
Running: valid_1065_filing
Running: irrelevant_question
Running: invalid_filters


,case_id,source_count,citations_valid,cited_numbers,generation_attempts,used_fallback,elapsed_seconds,sources_check,citations_check,case_passed,error
0,valid_1065_income,5,True,[],2,True,1.04,True,True,True,None
1,valid_1065_filing,5,True,[],2,True,0.45,True,True,True,None
2,irrelevant_question,0,False,[],0,True,0.01,True,True,True,None
3,invalid_filters,0,False,[],0,True,0.00,True,True,True,None



Cases passed: 4/4
Overall pass rate: 100.0%


### Inspect answers and diagnose failures

In [103]:
for case in evaluation_cases:
    case_id = case["case_id"]

    print("\n" + "=" * 80)
    print("CASE:", case_id)
    print("QUESTION:", case["question"])
    print("=" * 80)

    response = evaluation_responses.get(case_id)

    if response is None:
        error_rows = evaluation_results.loc[
            evaluation_results["case_id"] == case_id,
            "error",
        ]

        error_message = (
            error_rows.iloc[0]
            if not error_rows.empty
            else "Unknown evaluation error."
        )

        print("ERROR:", error_message)
        continue

    print("\nANSWER:")
    print(response["answer"])

    print("\nDIAGNOSTICS:")
    print("Sources retrieved:", len(response["sources"]))
    print(
        "Generation attempts:",
        response["generation_attempts"],
    )
    print(
        "Safe fallback used:",
        response["used_fallback"],
    )

    citation_check = response.get("citation_check")

    if citation_check is None:
        print("Citation validation: Not applicable")
    else:
        print(
            "Citations valid:",
            citation_check["citations_valid"],
        )
        print(
            "Citations used:",
            citation_check["cited_numbers"],
        )
        print(
            "Invalid citations:",
            citation_check["invalid_numbers"],
        )

    if not response["sources"].empty:
        print("\nTOP RETRIEVED SOURCES:")

        source_columns = [
            column
            for column in [
                "result_number",
                "similarity_score",
                "file_name",
                "return_type",
                "tax_year",
                "page_number",
            ]
            if column in response["sources"].columns
        ]

        display(
            response["sources"][source_columns]
        )


failed_cases = evaluation_results.loc[
    ~evaluation_results["case_passed"]
]

print("\n" + "=" * 80)

if failed_cases.empty:
    print("No failed evaluation cases.")
else:
    print("FAILED CASES:")
    display(
        failed_cases[
            [
                "case_id",
                "sources_check",
                "citations_check",
                "error",
            ]
        ]
    )


CASE: valid_1065_income
QUESTION: How is ordinary business income reported on Form 1065?

ANSWER:
The retrieved documents do not provide enough information.

DIAGNOSTICS:
Sources retrieved: 5
Generation attempts: 2
Safe fallback used: True
Citations valid: True
Citations used: []
Invalid citations: []

TOP RETRIEVED SOURCES:


,result_number,similarity_score,file_name,return_type,tax_year,page_number
0,1,0.6708,i1065--2025.pdf,1065,2025,4.0
1,2,0.6531,i1065s23--2025.pdf,1065,2025,13.0
2,3,0.6301,i1065sk1--2025.pdf,1065,2025,14.0
3,4,0.6211,i1065--2025.pdf,1065,2025,38.0
4,5,0.5906,i1065s23--2025.pdf,1065,2025,39.0



CASE: valid_1065_filing
QUESTION: Who must file Form 1065?

ANSWER:
The retrieved documents do not provide enough information.

DIAGNOSTICS:
Sources retrieved: 5
Generation attempts: 2
Safe fallback used: True
Citations valid: True
Citations used: []
Invalid citations: []

TOP RETRIEVED SOURCES:


,result_number,similarity_score,file_name,return_type,tax_year,page_number
0,1,0.5988,i1065--2025.pdf,1065,2025,30.0
1,2,0.5883,i1065x--2025.pdf,1065,2025,1.0
2,3,0.5753,i1065--2025.pdf,1065,2025,6.0
3,4,0.5486,f1065--2025.pdf,1065,2025,2.0
4,5,0.5408,p541--2025.pdf,1065,2025,19.0



CASE: irrelevant_question
QUESTION: What deduction applies to a business operating on Mars?

ANSWER:
No passages matched the selected form and tax-year filters.

DIAGNOSTICS:
Sources retrieved: 0
Generation attempts: 0
Safe fallback used: True
Citation validation: Not applicable

CASE: invalid_filters
QUESTION: How is business income reported?

ANSWER:
No passages matched the selected form and tax-year filters.

DIAGNOSTICS:
Sources retrieved: 0
Generation attempts: 0
Safe fallback used: True
Citation validation: Not applicable

No failed evaluation cases.


### Calculate final evaluation metrics

In [106]:
total_cases = len(evaluation_results)
passed_cases = int(
    evaluation_results["case_passed"].sum()
)

retrieval_checks_passed = int(
    evaluation_results["sources_check"].sum()
)

citation_checks_passed = int(
    evaluation_results["citations_check"].sum()
)

error_count = int(
    evaluation_results["error"].notna().sum()
)

fallback_count = int(
    evaluation_results["used_fallback"].sum()
)

average_latency = float(
    evaluation_results["elapsed_seconds"].mean()
)

final_metrics = pd.DataFrame(
    [
        {
            "metric": "Overall cases passed",
            "value": f"{passed_cases}/{total_cases}",
        },
        {
            "metric": "Overall pass rate",
            "value": f"{passed_cases / total_cases:.1%}",
        },
        {
            "metric": "Retrieval checks passed",
            "value": (
                f"{retrieval_checks_passed}/{total_cases}"
            ),
        },
        {
            "metric": "Citation checks passed",
            "value": (
                f"{citation_checks_passed}/{total_cases}"
            ),
        },
        {
            "metric": "Safe fallbacks used",
            "value": fallback_count,
        },
        {
            "metric": "Execution errors",
            "value": error_count,
        },
        {
            "metric": "Average latency",
            "value": f"{average_latency:.2f} seconds",
        },
    ]
)

display(final_metrics)

if passed_cases == total_cases and error_count == 0:
    print("\nFINAL STATUS: All automated evaluations passed.")
else:
    print(
        "\nFINAL STATUS: Review the failed cases "
        "before completing the notebook."
    )

,metric,value
0,Overall cases passed,4/4
1,Overall pass rate,100.0%
2,Retrieval checks passed,4/4
3,Citation checks passed,4/4
4,Safe fallbacks used,4
5,Execution errors,0
6,Average latency,0.38 seconds



FINAL STATUS: All automated evaluations passed.


### Save evaluation results

In [111]:
from pathlib import Path
import json


EVALUATION_DIR = Path(
    "/kaggle/working/tax_llm/evaluation"
)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)


# Save metric tables
evaluation_results.to_csv(
    EVALUATION_DIR / "evaluation_results.csv",
    index=False,
)

final_metrics.to_csv(
    EVALUATION_DIR / "final_metrics.csv",
    index=False,
)


# Save answers, diagnostics, and retrieved sources
detailed_results = []

for case in evaluation_cases:
    case_id = case["case_id"]
    response = evaluation_responses.get(case_id)

    if response is None:
        detailed_results.append(
            {
                "case_id": case_id,
                "question": case["question"],
                "error": "No response was produced.",
            }
        )
        continue

    sources = json.loads(
        response["sources"].to_json(
            orient="records"
        )
    )

    detailed_results.append(
        {
            "case_id": case_id,
            "question": case["question"],
            "answer": response["answer"],
            "generation_attempts": response[
                "generation_attempts"
            ],
            "used_fallback": response[
                "used_fallback"
            ],
            "citation_check": response.get(
                "citation_check"
            ),
            "sources": sources,
        }
    )


with open(
    EVALUATION_DIR / "detailed_results.json",
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        detailed_results,
        output_file,
        indent=2,
        ensure_ascii=False,
    )


all_passed = (
    bool(evaluation_results["case_passed"].all())
    and evaluation_results["error"].isna().all()
)

print("Saved evaluation files:")
for file_path in sorted(EVALUATION_DIR.iterdir()):
    print("-", file_path)

print(
    "\nNotebook status:",
    "COMPLETE" if all_passed else "REVIEW REQUIRED",
)

Saved evaluation files:
- /kaggle/working/tax_llm/evaluation/detailed_results.json
- /kaggle/working/tax_llm/evaluation/evaluation_results.csv
- /kaggle/working/tax_llm/evaluation/final_metrics.csv

Notebook status: COMPLETE


In [ ]:
# Interactive question-and-answer cell

FORM_FILTER = "1065"   # Change to None for all forms
YEAR_FILTER = 2025     # Change to None for all years
TOP_K = 5

print("Tax chatbot is ready.")
print("Type 'quit' to stop.\n")

while True:
    question = input("Ask a tax question: ").strip()

    if question.lower() in {"quit", "exit", "stop"}:
        print("Chatbot stopped.")
        break

    if not question:
        print("Please enter a question.\n")
        continue

    response = answer_tax_question(
        question=question,
        return_type=FORM_FILTER,
        tax_year=YEAR_FILTER,
        top_k=TOP_K,
    )

    print("\nANSWER")
    print(response["answer"])

    print("\nSUPPORTING SOURCES")
    if response["sources"].empty:
        print("No supporting passages were found.")
    else:
        display_retrieval_results(response["sources"])

    print("\n" + "=" * 70 + "\n")

Tax chatbot is ready.
Type 'quit' to stop.

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


Ask a tax question:  what is 1120 file



ANSWER
[1] or 2].

SUPPORTING SOURCES
[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 59
Similarity: 0.4040

to file Form 8990. Code AF. Excess business interest income. If the partnership is required to file Form 8990, it may determine it has excess business interest income. If so, enter the amount from Form 8990, Part II, line 37, for excess business interest income. Schedule K-1. Enter the partner’s amount of excess business interest income. The partner will enter the amount in column (g) of Form 8990, Schedule A, line 43, if the partner is required to file Form 8990. Instructions for Form 1065 (2025) 59


[2] i1065s23--2025.pdf — Form 1065 — tax year 2025 — page 13
Similarity: 0.4040

Form 1116 only requires reporting of total gross income from foreign sources by separate category. Therefore, those required to file Form 1116 will report Schedule K-3, Part II, Section 1, line 24, by country on their Form 1116, Part I, line 1a. Section 1 also generally follows the types of gr

Ask a tax question:  Who must file Form 1065?



ANSWER
The retrieved documents do not provide enough information.

SUPPORTING SOURCES
[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 30
Similarity: 0.5988

file Form 1065 except for the year of election. If an election out of subchapter K is being made for the tax year, answer “Yes” and attach a statement that contains: • The names, addresses, and identification numbers of all the members of the organization; • A statement that the organization qualifies under Regulations section 1.761-2(a), paragraph (1), and either paragraph (2) or (3); • A statement that all members of the organization elect to exclude the organization from all of subchapter K; and • A statement indicating the availability of the agreement under which the organization operates (or for an oral agreement, from whom the provisions of the agreement may be obtained). For calendar-year organizations, Form 1065 must be filed by March 15 following the close of the first calendar year for which the section 761(a) el

Ask a tax question:  When is Form 1065 due?



ANSWER
The retrieved documents do not provide enough information.

SUPPORTING SOURCES
[1] i1065--2025.pdf — Form 1065 — tax year 2025 — page 5
Similarity: 0.5697

of Time To File Certain Business Income Tax, Information, and Other Returns, to request an extension of time to file. File Form 7004 by the regular due date of the partnership return. Form 7004 can be electronically filed. See the Instructions for Form 7004. Period Covered The 2025 Form 1065 is an information return for calendar year 2025 and fiscal years that begin in 2025 and end in 2026. For a fiscal year or a short tax year, fill in the tax year space at the top of Form 1065 and each Schedule K-1 or K-3, if applicable. Instructions for Form 1065 (2025) 5


[2] i1065--2025.pdf — Form 1065 — tax year 2025 — page 30
Similarity: 0.5687

file Form 1065 except for the year of election. If an election out of subchapter K is being made for the tax year, answer “Yes” and attach a statement that contains: • The names, addresses, a

Ask a tax question:  How is ordinary business income reported?



ANSWER
The retrieved documents do not provide enough information.

SUPPORTING SOURCES
[1] i1065sk1--2025.pdf — Form 1065 — tax year 2025 — page 14
Similarity: 0.6118

it has attached a statement providing additional information. For those informational items that can’t be reported as a single dollar amount, the partnership will enter an asterisk (*) in the left column and enter “STMT” in the dollar amount entry space to indicate the information is provided on an attached statement. Income (Loss) Box 1. Ordinary Business Income (Loss) The amount reported in box 1 is your share of the ordinary income (loss) from trade or business activities of the partnership. Generally, where you report this amount on Form 1040 or 1040-SR depends on whether the amount is from an activity that’s a passive activity to you. If you’re an individual partner filing a 2025 Form 1040 or 1040-SR, find your situation below and report your box 1 income (loss) as instructed, after applying the basis and at-risk li

### 